### **Fatigue risk**

In [ ]:
import re
import numpy as np
import pandas as pd
from scipy.stats import norm

# ============================================================
# FDOT / AASHTO FATIGUE SCREENING (TM2) — USING TABLE 9.3
# ============================================================
# Uses Table 9.3 fatigue model parameters:
#   - R_y^M (mean/median factor)
#   - R_y^E (evaluation factor)
#   - K_y (ksi^3)
#
# Computes:
#   1) ADTT = ADT * Truck%
#   2) ADTT_SL = ADTT * SLF (screening SLF ~ 1/lanes or 1.0)
#   3) AM, AE (years) using:
#        - Eq 3.4 if g == 1
#        - Eq 3.5 / 3.6 if g != 1 (optional)
#   4) Pf(A) = Phi( ln(A/AM) / ln(AM/AE) )  (Eq 3.11)
#
# Outputs:
#   fatigue_all_ranked.csv
#   fatigue_top_25.csv
# ============================================================

# ----------------------------
# CONFIG
# ----------------------------
INPUT_CSV = "tx25-final.csv"
CURRENT_YEAR = 2025

# Stress range S (ksi) and cycles per truck passage C (screening)
S_KSI = 5.0
C_CYC = 1.0

# Traffic growth factor g and horizon T1 (only used if g != 1)
GROWTH_G = 1.0   # set > 1.0 if your procedure uses growth
T1_YEARS  = 75.0 # used only when g != 1

# Choose one fatigue category from: A, B, Bp, C, Cp, D, E, Ep
DETAIL = "D"

# ----------------------------
# TABLE 9.3: Fatigue model parameters (CORRECT VALUES)
# ----------------------------
DETAIL_PARAMS = {
    "A":  {"RyM": 2.8, "RyE": 1.7, "Ky": 2.50e10},
    "B":  {"RyM": 2.0, "RyE": 1.4, "Ky": 1.20e10},
    "Bp": {"RyM": 2.4, "RyE": 1.5, "Ky": 6.10e9},  # B'
    "C":  {"RyM": 1.3, "RyE": 1.2, "Ky": 4.39e9},
    "Cp": {"RyM": 1.3, "RyE": 1.2, "Ky": 4.39e9},  # C'
    "D":  {"RyM": 1.6, "RyE": 1.3, "Ky": 2.20e9},
    "E":  {"RyM": 1.6, "RyE": 1.3, "Ky": 1.10e9},
    "Ep": {"RyM": 2.5, "RyE": 1.6, "Ky": 3.90e8},  # E'
}

# Cost placeholders (only if you want expected cost risk; optional)
DO_COST_RISK = False
DEFAULT_CREPAIR = 250_000.0
DEFAULT_USERCOST = 100_000.0
FC_MULTIPLIER = 2.0

# Rank output by:
RANK_BY = "Pf_fatigue"  # or "FatigueRisk_expected_total" if DO_COST_RISK=True

# ----------------------------
# HELPERS
# ----------------------------
def pick_col(df: pd.DataFrame, patterns, required=False):
    cols = list(df.columns)
    for pat in patterns:
        rx = re.compile(pat, flags=re.IGNORECASE)
        for c in cols:
            if rx.search(c):
                return c
    if required:
        raise KeyError(f"Missing required column. Tried patterns: {patterns}")
    return None

def safe_num(x, default=0.0):
    return pd.to_numeric(x, errors="coerce").fillna(default)

def nbi_lat_to_dd(x):
    if pd.isna(x): return np.nan
    try: v = float(x)
    except: return np.nan
    # already decimal degrees?
    if abs(v) <= 90.999999:
        return v
    v = int(round(v))
    d = v // 1_000_000
    m = (v // 10_000) % 100
    s = (v % 10_000) / 100.0
    return d + m/60.0 + s/3600.0

def nbi_lon_to_dd(x):
    if pd.isna(x): return np.nan
    try: v = float(x)
    except: return np.nan
    # already decimal degrees?
    if abs(v) <= 180.999999:
        return v if v <= 0 else -v
    v = int(round(v))
    d = v // 1_000_000
    m = (v // 10_000) % 100
    s = (v % 10_000) / 100.0
    return -(d + m/60.0 + s/3600.0)

def fatigue_life_years(Ry, Ky, adtt_sl, S=S_KSI, C=C_CYC, g=GROWTH_G, T1=T1_YEARS):
    """
    FDOT forms:
      - g == 1: Eq 3.4  Y = (Ry*Ky)/(365*C*ADTT_SL*S^3)
      - g != 1: Eq 3.5 / 3.6 piecewise
    """
    if (adtt_sl is None) or (adtt_sl <= 0):
        return np.nan

    base = (Ry * Ky) / (365.0 * C * adtt_sl * (S ** 3))
    if (not np.isfinite(base)) or (base <= 0):
        return np.nan

    if g is None or abs(g - 1.0) < 1e-12:
        return float(base)

    g = float(g)
    if g <= 0:
        return float(base)

    # Eq 3.5
    Y1 = np.log((g - 1.0) * base + 1.0) / np.log(g)
    if np.isfinite(Y1) and (Y1 <= T1):
        return float(Y1)

    # Eq 3.6
    Y2 = T1 + (np.log((g - 1.0) * base + 1.0) - np.log((g - 1.0) * T1 + 1.0)) / np.log(g)
    return float(Y2) if np.isfinite(Y2) else np.nan

def fatigue_pf(age_years, adtt_sl, Ky, RyM, RyE, g=GROWTH_G):
    if age_years is None or age_years <= 0 or adtt_sl is None or adtt_sl <= 0:
        return 0.0, np.nan, np.nan

    AM = fatigue_life_years(RyM, Ky, adtt_sl, g=g)
    AE = fatigue_life_years(RyE, Ky, adtt_sl, g=g)

    if (not np.isfinite(AM)) or (not np.isfinite(AE)) or (AM <= 0) or (AE <= 0) or (AM <= AE):
        return 0.0, AM, AE

    z = np.log(age_years / AM) / np.log(AM / AE)
    pf = float(norm.cdf(z))
    return max(0.0, min(1.0, pf)), AM, AE

# ----------------------------
# LOAD
# ----------------------------
df = pd.read_csv(INPUT_CSV, low_memory=False)

# ----------------------------
# COORDINATES (optional)
# ----------------------------
if ("LAT_016" in df.columns) and ("LONG_017" in df.columns):
    lat_dd = df["LAT_016"].apply(nbi_lat_to_dd)
    lon_dd = df["LONG_017"].apply(nbi_lon_to_dd)
    df["coordinates (lat, lon)"] = (
        pd.Series(lat_dd).round(6).astype("string") + ", " +
        pd.Series(lon_dd).round(6).astype("string")
    )
else:
    df["coordinates (lat, lon)"] = pd.NA

# ----------------------------
# TRAFFIC: ADT + TRUCK% -> ADTT
# ----------------------------
adt_col = pick_col(df, [r"^ADT_029$", r"\bADT\b"], required=True)
trk_col = pick_col(df, [r"^PERCENT_ADT_TRUCK_109$", r"PERCENT.*TRUCK", r"TRUCK.*PCT"], required=True)

df["ADT"] = safe_num(df[adt_col], 0.0)
df["TRUCK_PCT"] = safe_num(df[trk_col], 0.0)
df["ADTT"] = df["ADT"] * (df["TRUCK_PCT"] / 100.0)

# ----------------------------
# AGE (built vs reconstructed)
# ----------------------------
yb_col = pick_col(df, [r"YEAR_BUILT_027", r"YEAR.*BUILT"], required=True)
yr_col = pick_col(df, [r"YEAR_RECONSTRUCTED_106", r"YEAR.*RECON", r"REHAB.*YEAR"], required=False)

yb = safe_num(df[yb_col], np.nan)
yr = safe_num(df[yr_col], 0.0) if yr_col else pd.Series(0.0, index=df.index)

use_yr = (yr > 1800) & (yr <= CURRENT_YEAR)
effective_year = np.where(use_yr, np.maximum(yb.fillna(0), yr), yb)
df["EFFECTIVE_YEAR"] = pd.to_numeric(effective_year, errors="coerce")

df["AGE"] = (CURRENT_YEAR - df["EFFECTIVE_YEAR"]).astype("float64")
df.loc[(df["AGE"].isna()) | (df["AGE"] <= 0), "AGE"] = 1.0

# ----------------------------
# SINGLE-LANE FACTOR (SLF) and ADTT_SL
# ----------------------------
lanes_col = pick_col(df, [r"LANES_ON_028A", r"LANES.*028A", r"\bLANE"], required=False)
if lanes_col:
    lanes = safe_num(df[lanes_col], 1.0).clip(lower=1.0)
    df["SLF"] = (1.0 / lanes).clip(lower=0.1, upper=1.0)  # screening approximation
else:
    df["SLF"] = 1.0

df["ADTT_SL"] = df["ADTT"] * df["SLF"]

# ----------------------------
# OPTIONAL: STEEL FILTER (if present) — else compute for all
# ----------------------------
steel_flag = pd.Series(False, index=df.index)

if "STRUCTURE_KIND_043A" in df.columns:
    sc = safe_num(df["STRUCTURE_KIND_043A"], np.nan)
    steel_flag |= sc.isin([3, 4])

for col in ["MAIN_UNIT_MATL_043A", "SUPERSTRUCTURE_MATL_043B", "STRUCTURE_TYPE_043B"]:
    if col in df.columns:
        steel_flag |= df[col].astype(str).str.lower().str.contains("steel", na=False)

work_df = df[steel_flag].copy()
if work_df.empty:
    work_df = df.copy()
    work_df["NOTE"] = "Steel filter not applied (steel column not found). Computed for all bridges."
else:
    work_df["NOTE"] = ""

# ----------------------------
# APPLY TABLE 9.3 PARAMETERS
# ----------------------------
if DETAIL not in DETAIL_PARAMS:
    raise ValueError(f"DETAIL='{DETAIL}' not recognized. Use one of: {list(DETAIL_PARAMS.keys())}")

Ky  = float(DETAIL_PARAMS[DETAIL]["Ky"])
RyM = float(DETAIL_PARAMS[DETAIL]["RyM"])
RyE = float(DETAIL_PARAMS[DETAIL]["RyE"])

tmp = work_df.apply(lambda r: fatigue_pf(r["AGE"], r["ADTT_SL"], Ky, RyM, RyE, g=GROWTH_G), axis=1)
work_df["Pf_fatigue"] = tmp.apply(lambda x: x[0])
work_df["AM_years"]   = tmp.apply(lambda x: x[1])
work_df["AE_years"]   = tmp.apply(lambda x: x[2])

# ----------------------------
# OPTIONAL: EXPECTED COST RISK (if you want)
# ----------------------------
if DO_COST_RISK:
    fc_col = pick_col(work_df, [r"FRACTURE", r"\bFC\b", r"FRACTURE.*CRITICAL"], required=False)
    crep_col = pick_col(work_df, [r"\bC_REP\b", r"REPAIR.*COST", r"CREPAIR", r"AGENCY.*COST"], required=False)
    ucost_col = pick_col(work_df, [r"USER.*COST", r"\bC_U\b", r"DETOUR.*COST"], required=False)

    work_df["C_repair"] = safe_num(work_df[crep_col], DEFAULT_CREPAIR).clip(lower=0.0) if crep_col else DEFAULT_CREPAIR
    work_df["C_user"]   = safe_num(work_df[ucost_col], DEFAULT_USERCOST).clip(lower=0.0) if ucost_col else DEFAULT_USERCOST

    if fc_col:
        s = work_df[fc_col].astype(str).str.lower()
        work_df["FC_flag"] = s.str.contains("y|yes|1|true|fc", regex=True, na=False).astype(int)
    else:
        work_df["FC_flag"] = 0

    work_df["C_user_adj"] = np.where(work_df["FC_flag"] == 1, work_df["C_user"] * FC_MULTIPLIER, work_df["C_user"])
    work_df["AgencyCost_expected"] = work_df["Pf_fatigue"] * work_df["C_repair"]
    work_df["UserCost_expected"]   = work_df["Pf_fatigue"] * work_df["C_user_adj"]
    work_df["FatigueRisk_expected_total"] = work_df["AgencyCost_expected"] + work_df["UserCost_expected_total"] \
        if "UserCost_expected_total" in work_df.columns else (work_df["AgencyCost_expected"] + work_df["UserCost_expected"])

# ----------------------------
# EXPORT
# ----------------------------
work_df_sorted = work_df.sort_values(RANK_BY, ascending=False)
work_df_sorted.to_csv("fatigue_all_ranked.csv", index=False)
work_df_sorted.head(25).to_csv("fatigue_top_25.csv", index=False)

print("Saved:")
print(" - fatigue_all_ranked.csv")
print(" - fatigue_top_25.csv")
print(f"DETAIL={DETAIL} | RyM={RyM} | RyE={RyE} | Ky={Ky:.3e} (ksi^3)")
print(f"g={GROWTH_G} | T1={T1_YEARS} | S={S_KSI} ksi | C={C_CYC}")
print(f"Ranking by: {RANK_BY}")
if isinstance(work_df_sorted.get("NOTE", pd.Series([""])).iloc[0], str) and work_df_sorted["NOTE"].iloc[0]:
    print("NOTE:", work_df_sorted["NOTE"].iloc[0])


Saved:
 - fatigue_all_ranked.csv
 - fatigue_top_25.csv
DETAIL=D | RyM=1.6 | RyE=1.3 | Ky=2.200e+09 (ksi^3)
g=1.0 | T1=75.0 | S=5.0 ksi | C=1.0
Ranking by: Pf_fatigue


### **wildfire risk**

In [ ]:
# ============================================================
# FDOT-STYLE WILDFIRE RISK (Texas) — TM2 §4.2.1 ALIGNED
# USING:
#   - TX25.csv (NBI bridge inventory)
#   - Wildfire CSV (point events with LAT/LON + YEAR + ID)
#
# FIXED:
#   ✅ Correct NBI coordinate conversion (LAT_016 = DDMMSSss, LONG_017 = DDDMMSSss)
#   ✅ Texas sanity bounding-box to prevent Africa/ocean points
#   ✅ Design vulnerability V_des uses FDOT Table 4.3 (score s_des 0..5) then:
#        V_des = (s_des + 1)/3
#
# REMINDER:
#   Step 6 / Element vulnerability multiplier V_elem = 1.0 (forced)
#
# Output keeps ALL original bridge columns and adds:
#   coordinates (lat, lon),
#   N_wildfires_within_1mi, T_years_wildfire_obs, lambda_wildfire, p_wildfire,
#   sdes_wildfire_raw_0to5, V_age_wildfire, V_des_wildfire, V_elem_wildfire,
#   Risk_wildfire, plus a design mapping note column.
# ============================================================

import numpy as np
import pandas as pd
from sklearn.neighbors import BallTree

# -----------------------------
# USER SETTINGS
# -----------------------------
BRIDGE_CSV   = "TX-25.csv"
WILDFIRE_CSV = "Texas_Wildfire_Data.csv"  # <-- change as needed
OUT_CSV      = "wildfire.csv"

BUFFER_MILES   = 1.0
EARTH_RADIUS_M = 6371000.0
BUFFER_METERS  = BUFFER_MILES * 1609.344
BUFFER_RADIANS = BUFFER_METERS / EARTH_RADIUS_M

# Age vulnerability (same style used across TM2 hazards)
REF_YEAR = 1992
AGE_K    = 35.0

# Observation window T
FORCE_T_YEARS = None  # None => infer from wildfire min/max year

# Wildfire CSV column names
WF_LAT_COL  = "LATITUDE"
WF_LON_COL  = "LONGITUDE"
WF_YEAR_COL = "FIRE_YEAR"
WF_ID_CANDIDATES = ["FOD_ID", "FPA_ID", "OBJECTID"]

# Texas sanity bounds (prevents Africa/ocean)
TX_LAT_MIN, TX_LAT_MAX = 25.0, 37.5
TX_LON_MIN, TX_LON_MAX = -107.5, -93.0

# -----------------------------
# HELPERS
# -----------------------------
def pick_first_existing(df: pd.DataFrame, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

def safe_numeric(s):
    return pd.to_numeric(s, errors="coerce")

def format_coord(lat, lon, decimals=6):
    if pd.isna(lat) or pd.isna(lon):
        return ""
    return f"{lat:.{decimals}f}, {lon:.{decimals}f}"

def nbi_lat_to_dd(x):
    """
    NBI LAT_016 typically stored as DDMMSSss (integer-like).
    If already decimal degrees (-90..90), return as-is.
    """
    if pd.isna(x): return np.nan
    try: v = float(x)
    except: return np.nan

    # already decimal latitude
    if -90.0 <= v <= 90.0:
        return v

    try:
        v = int(abs(v))
        d = v // 1_000_000
        m = (v // 10_000) % 100
        s = (v % 10_000) / 100.0
        return d + m/60.0 + s/3600.0
    except:
        return np.nan

def nbi_lon_to_dd(x):
    """
    NBI LONG_017 typically stored as DDDMMSSss (integer-like).
    If already decimal degrees (-180..180), keep; enforce West negative.
    """
    if pd.isna(x): return np.nan
    try: v = float(x)
    except: return np.nan

    # already decimal longitude
    if -180.0 <= v <= 180.0:
        return v if v <= 0 else -v

    try:
        v_abs = int(abs(v))
        d = v_abs // 1_000_000
        m = (v_abs // 10_000) % 100
        s = (v_abs % 10_000) / 100.0
        lon = d + m/60.0 + s/3600.0
        # West negative (Texas)
        return -lon
    except:
        return np.nan

def compute_age_multiplier(df, year_built_col):
    y = safe_numeric(df[year_built_col]).fillna(1900)
    v_age = 1.0 + (REF_YEAR - y) / AGE_K
    return v_age.clip(lower=1.0)

# -----------------------------
# FDOT TABLE 4.3 — DESIGN VULNERABILITY SCORE s_des (0..5)
# Rows = NBI Item 43B "type of design construction" (common coding)
# Cols = material class (we infer from NBI item(s))
#
# We implement the filled cells exactly as shown in your Table 4.3 image:
#   - Concrete / Concrete Continuous => 3 for many common girder types
#   - Steel / Steel Continuous => 4 for many steel superstructures (stringer/truss/etc.)
#   - Prestressed / Prestressed Continuous => 3 for many beam/girder types
#   - Wood/Timber => 5 for slab/stringer/girder&floorbeam
#   - Masonry => 3 for arch deck/thru
# Cells not shown in the table are treated as "not specified" => fallback score (3)
# and flagged via a note column so you can review.
# -----------------------------

# Map NBI 43B code -> Table 4.3 row name
# (This is the standard NBI-style list; if TX uses different, adjust here.)
DESIGN43B_TO_ROW = {
    1:  "Slab",
    2:  "Stringer/MultiBeam/Girder",
    3:  "Girder & Floorbeam",
    4:  "Tee Beam",
    5:  "Box Beam or Girders - Multiple",
    6:  "Box Beam/Girders - Single or Spread",
    7:  "Frame (except frame culverts)",
    8:  "Orthotropic",
    9:  "Truss - Deck",
    10: "Truss - Thru",
    11: "Arch - Deck",
    12: "Arch - Thru",
    13: "Suspension",
    14: "Stayed Girder",
    15: "Movable - Lift",
    16: "Movable - Bascule",
    17: "Movable - Swing",
    18: "Tunnel",
    19: "Culvert",
    20: "Mixed Types",
    21: "Segmental Box Girder",
    22: "Channel Beam",
}

# Material classes used by Table 4.3 columns
MAT_CONC   = "Concrete"
MAT_CONC_C = "Concrete Continuous"
MAT_STEEL  = "Steel"
MAT_STEEL_C= "Steel Continuous"
MAT_PS     = "Prestressed concrete"
MAT_PS_C   = "Prestressed concrete continuous"
MAT_WOOD   = "Wood or Timber"
MAT_MASON  = "Masonry"
MAT_AL     = "Aluminum/Wrought/Cast Iron"

# Table 4.3 filled cells (row, material) -> s_des
T43 = {}

def _set(row, mat, val):
    T43[(row, mat)] = float(val)

# Slab
_set("Slab", MAT_CONC,   3)
_set("Slab", MAT_CONC_C, 3)
_set("Slab", MAT_PS,     3)
_set("Slab", MAT_PS_C,   3)
_set("Slab", MAT_WOOD,   5)

# Stringer/MultiBeam/Girder
for m in [MAT_CONC, MAT_CONC_C, MAT_PS, MAT_PS_C]:
    _set("Stringer/MultiBeam/Girder", m, 3)
_set("Stringer/MultiBeam/Girder", MAT_STEEL,   4)
_set("Stringer/MultiBeam/Girder", MAT_STEEL_C, 4)
_set("Stringer/MultiBeam/Girder", MAT_WOOD,    5)

# Girder & Floorbeam
for m in [MAT_CONC, MAT_CONC_C, MAT_PS, MAT_PS_C]:
    _set("Girder & Floorbeam", m, 3)
_set("Girder & Floorbeam", MAT_STEEL,   4)
_set("Girder & Floorbeam", MAT_STEEL_C, 4)
_set("Girder & Floorbeam", MAT_WOOD,    5)

# Tee Beam
for m in [MAT_CONC, MAT_CONC_C, MAT_PS, MAT_PS_C]:
    _set("Tee Beam", m, 3)
_set("Tee Beam", MAT_STEEL,   4)
_set("Tee Beam", MAT_STEEL_C, 4)

# Box Beam / Girders - Multiple
for m in [MAT_CONC, MAT_CONC_C, MAT_PS, MAT_PS_C]:
    _set("Box Beam or Girders - Multiple", m, 3)

# Box Beam / Girders - Single or Spread
for m in [MAT_CONC, MAT_CONC_C, MAT_PS, MAT_PS_C]:
    _set("Box Beam/Girders - Single or Spread", m, 3)

# Truss - Deck / Thru
_set("Truss - Deck", MAT_STEEL,   4)
_set("Truss - Deck", MAT_STEEL_C, 4)
_set("Truss - Thru", MAT_STEEL,   4)
_set("Truss - Thru", MAT_STEEL_C, 4)

# Arch - Deck / Thru (masonry shown as 3)
_set("Arch - Deck", MAT_MASON, 3)
_set("Arch - Thru", MAT_MASON, 3)

# Suspension / Stayed
_set("Suspension",   MAT_STEEL,   4)
_set("Suspension",   MAT_STEEL_C, 4)
_set("Stayed Girder",MAT_STEEL,   4)
_set("Stayed Girder",MAT_STEEL_C, 4)

# Movable types (lift/bascule/swing)
for row in ["Movable - Lift", "Movable - Bascule", "Movable - Swing"]:
    _set(row, MAT_CONC,   3)
    _set(row, MAT_CONC_C, 3)
    _set(row, MAT_STEEL,  4)
    _set(row, MAT_STEEL_C,4)

# Segmental Box Girder
for m in [MAT_CONC, MAT_CONC_C, MAT_PS, MAT_PS_C]:
    _set("Segmental Box Girder", m, 3)

# Channel Beam
_set("Channel Beam", MAT_STEEL,   4)
_set("Channel Beam", MAT_STEEL_C, 4)

# ----- Material inference from NBI -----
def infer_material_class(df: pd.DataFrame):
    """
    Try to infer Table 4.3 column class for each bridge using available NBI fields.
    Priority:
      1) Text columns containing keywords (prestress, steel, timber, masonry, aluminum, concrete)
      2) Numeric STRUCTURE_KIND_043A style codes
    Returns: material_class (Series of strings)
    """
    n = len(df)
    mat = pd.Series([None]*n, index=df.index, dtype="object")

    # Text-based detection from any likely material columns
    text_cols = [c for c in ["MAIN_UNIT_MATL_043A", "SUPERSTRUCTURE_MATL_043B", "STRUCTURE_TYPE_043B", "STRUCTURE_KIND_043A"]
                 if c in df.columns]

    if text_cols:
        blob = pd.Series([""]*n, index=df.index, dtype="string")
        for c in text_cols:
            blob = blob + " " + df[c].astype("string").fillna("")

        t = blob.str.lower()

        mat = mat.mask(mat.isna() & t.str.contains("prestress"), MAT_PS)  # default to prestressed (non-continuous)
        mat = mat.mask(mat.isna() & t.str.contains("steel"), MAT_STEEL)
        mat = mat.mask(mat.isna() & t.str.contains("timber|wood"), MAT_WOOD)
        mat = mat.mask(mat.isna() & t.str.contains("masonry|brick|stone"), MAT_MASON)
        mat = mat.mask(mat.isna() & t.str.contains("aluminum|wrought|cast iron"), MAT_AL)
        mat = mat.mask(mat.isna() & t.str.contains("concrete"), MAT_CONC)

        # If text indicates "continuous"
        mat = mat.mask(t.str.contains("continuous") & (mat == MAT_CONC),   MAT_CONC_C)
        mat = mat.mask(t.str.contains("continuous") & (mat == MAT_STEEL),  MAT_STEEL_C)
        mat = mat.mask(t.str.contains("continuous") & (mat == MAT_PS),     MAT_PS_C)

    # Numeric-code fallback (common NBI-style codes)
    if "STRUCTURE_KIND_043A" in df.columns:
        a = safe_numeric(df["STRUCTURE_KIND_043A"])
        # If codes appear to be standard 1..8 (no prestressed), map those
        # 1=Concrete, 2=Concrete Continuous, 3=Steel, 4=Steel Continuous, 5=Timber, 6=Masonry, 7=Aluminum, 8=Other
        mat = mat.mask(mat.isna() & a.isin([1]), MAT_CONC)
        mat = mat.mask(mat.isna() & a.isin([2]), MAT_CONC_C)
        mat = mat.mask(mat.isna() & a.isin([3]), MAT_STEEL)
        mat = mat.mask(mat.isna() & a.isin([4]), MAT_STEEL_C)
        mat = mat.mask(mat.isna() & a.isin([5]), MAT_WOOD)
        mat = mat.mask(mat.isna() & a.isin([6]), MAT_MASON)
        mat = mat.mask(mat.isna() & a.isin([7]), MAT_AL)

        # If TX has extended coding where 5/6 represent prestressed, you can enable this:
        # mat = mat.mask(mat.isna() & a.isin([5]), MAT_PS)
        # mat = mat.mask(mat.isna() & a.isin([6]), MAT_PS_C)

    # Default if still unknown
    mat = mat.fillna(MAT_CONC)  # conservative baseline (many TX bridges are concrete)
    return mat

def compute_sdes_vdes_from_table43(df: pd.DataFrame, col_43b: str):
    """
    Uses Table 4.3 mapping to produce:
      - row_name (design type row)
      - material_class (Table 4.3 column)
      - sdes_raw_0to5
      - V_des = (sdes+1)/3
      - note if fallback was used
    """
    b = safe_numeric(df[col_43b])
    row = b.map(DESIGN43B_TO_ROW).fillna("Unknown")
    mat = infer_material_class(df)

    sdes = pd.Series(np.nan, index=df.index, dtype="float64")
    note = pd.Series([""] * len(df), index=df.index, dtype="string")  # ✅ FIXED

    # lookup
    for idx in df.index:
        key = (row.loc[idx], mat.loc[idx])
        if key in T43:
            sdes.loc[idx] = T43[key]
        else:
            sdes.loc[idx] = 3.0
            note.loc[idx] = f"Fallback sdes=3 (no Table4.3 cell for row='{row.loc[idx]}' col='{mat.loc[idx]}')"

    vdes = (sdes + 1.0) / 3.0
    return row, mat, sdes, vdes, note

# -----------------------------
# 1) LOAD DATA
# -----------------------------
bridges = pd.read_csv(BRIDGE_CSV, low_memory=False, dtype={"STRUCTURE_NUMBER_008": str})
fires   = pd.read_csv(WILDFIRE_CSV, low_memory=False)

# -----------------------------
# 2) REQUIRED COLUMNS
# -----------------------------
BR_LAT_COL = "LAT_016"  if "LAT_016"  in bridges.columns else pick_first_existing(bridges, ["LAT", "LATITUDE"])
BR_LON_COL = "LONG_017" if "LONG_017" in bridges.columns else pick_first_existing(bridges, ["LON", "LONGITUDE"])
if BR_LAT_COL is None or BR_LON_COL is None:
    raise ValueError("Bridge file missing LAT/LON columns. Expected LAT_016 and LONG_017.")

for c in [WF_LAT_COL, WF_LON_COL, WF_YEAR_COL]:
    if c not in fires.columns:
        raise ValueError(f"Wildfire file missing required column '{c}'.")

WF_ID_COL = pick_first_existing(fires, WF_ID_CANDIDATES)
if WF_ID_COL is None:
    raise ValueError(f"Wildfire file missing unique ID column. Tried {WF_ID_CANDIDATES}")

YEAR_BUILT_COL = pick_first_existing(bridges, ["YEAR_BUILT_027", "YEAR_BUILT", "YR_BUILT"])
if YEAR_BUILT_COL is None:
    raise ValueError("Could not find YEAR_BUILT column (expected YEAR_BUILT_027).")

# NBI 43B (superstructure design type) is required for Table 4.3 rows
COL_43B = pick_first_existing(bridges, ["STRUCTURE_TYPE_043B", "TYPE_DESIGN_043B", "ITEM_43B", "NBI_43B"])
if COL_43B is None:
    raise ValueError("Could not find NBI Item 43B design-type column (need it for Table 4.3 rows).")

# -----------------------------
# 3) COORDINATES (FIXED) + TEXAS SANITY FILTER
# -----------------------------
bridges["_lat_dd"] = bridges[BR_LAT_COL].apply(nbi_lat_to_dd)
bridges["_lon_dd"] = bridges[BR_LON_COL].apply(nbi_lon_to_dd)

tx_ok = bridges["_lat_dd"].between(TX_LAT_MIN, TX_LAT_MAX) & bridges["_lon_dd"].between(TX_LON_MIN, TX_LON_MAX)
bridges.loc[~tx_ok, ["_lat_dd", "_lon_dd"]] = np.nan

bridges["coordinates (lat, lon)"] = [
    format_coord(lat, lon, decimals=6) for lat, lon in zip(bridges["_lat_dd"], bridges["_lon_dd"])
]

# -----------------------------
# 4) PREP FIRE DATA (TX only, dedupe, radians)
# -----------------------------
if "STATE" in fires.columns:
    fires = fires[fires["STATE"].astype(str).str.upper().eq("TX")].copy()

fires[WF_LAT_COL]  = safe_numeric(fires[WF_LAT_COL])
fires[WF_LON_COL]  = safe_numeric(fires[WF_LON_COL])
fires[WF_YEAR_COL] = safe_numeric(fires[WF_YEAR_COL])

fires = fires.dropna(subset=[WF_LAT_COL, WF_LON_COL, WF_YEAR_COL])
fires = fires.drop_duplicates(subset=[WF_ID_COL])

min_year = int(fires[WF_YEAR_COL].min())
max_year = int(fires[WF_YEAR_COL].max())
T_years = FORCE_T_YEARS if FORCE_T_YEARS is not None else (max_year - min_year + 1)
if T_years <= 0:
    raise ValueError("Invalid wildfire observation window T. Check FIRE_YEAR values.")

fires["_lat_rad"] = np.deg2rad(fires[WF_LAT_COL].values)
fires["_lon_rad"] = np.deg2rad(fires[WF_LON_COL].values)
fire_points = np.vstack([fires["_lat_rad"].values, fires["_lon_rad"].values]).T

tree = BallTree(fire_points, metric="haversine")

# -----------------------------
# 5) COUNT FIRES WITHIN 1 MILE PER BRIDGE
# -----------------------------
bridges["N_wildfires_within_1mi"] = 0

valid_br = bridges.dropna(subset=["_lat_dd", "_lon_dd"]).copy()
if len(valid_br) > 0:
    br_points = np.vstack([
        np.deg2rad(valid_br["_lat_dd"].values),
        np.deg2rad(valid_br["_lon_dd"].values)
    ]).T

    idx_arrays = tree.query_radius(br_points, r=BUFFER_RADIANS)
    Ni = np.array([len(a) for a in idx_arrays], dtype=np.int32)
    bridges.loc[valid_br.index, "N_wildfires_within_1mi"] = Ni

# -----------------------------
# 6) LIKELIHOOD (Poisson)
# -----------------------------
bridges["T_years_wildfire_obs"] = T_years
bridges["lambda_wildfire"] = bridges["N_wildfires_within_1mi"] / float(T_years)
bridges["p_wildfire"] = 1.0 - np.exp(-bridges["lambda_wildfire"].astype(float))

# -----------------------------
# 7) VULNERABILITY MULTIPLIERS
# -----------------------------
bridges["V_elem_wildfire"] = 1.0  # forced per your instruction

bridges["V_age_wildfire"] = compute_age_multiplier(bridges, YEAR_BUILT_COL)

# Table 4.3 design vulnerability
row_name, mat_class, sdes_raw, vdes, sdes_note = compute_sdes_vdes_from_table43(bridges, COL_43B)
bridges["wildfire_design_row_43B"] = row_name
bridges["wildfire_material_class"] = mat_class
bridges["sdes_wildfire_raw_0to5"]  = sdes_raw
bridges["V_des_wildfire"]          = vdes
bridges["sdes_wildfire_note"]      = sdes_note

# -----------------------------
# 8) RISK INDEX
# -----------------------------
bridges["Risk_wildfire"] = (
    bridges["p_wildfire"].astype(float)
    * bridges["V_elem_wildfire"].astype(float)
    * bridges["V_age_wildfire"].astype(float)
    * bridges["V_des_wildfire"].astype(float)
)

# -----------------------------
# 9) SAVE OUTPUT (KEEP ALL ORIGINAL COLUMNS)
# -----------------------------
bridges_out = bridges.drop(columns=[c for c in ["_lat_dd", "_lon_dd"] if c in bridges.columns], errors="ignore")
bridges_out.to_csv(OUT_CSV, index=False)

print("✅ Done!")
print(f"Wildfire years observed: {min_year}–{max_year}  (T = {T_years} years)")
print(f"Saved: {OUT_CSV}")

# OPTIONAL: show top 20
show_cols = [
    "STRUCTURE_NUMBER_008",
    "coordinates (lat, lon)",
    "N_wildfires_within_1mi",
    "lambda_wildfire",
    "p_wildfire",
    "V_elem_wildfire",
    "V_age_wildfire",
    "wildfire_design_row_43B",
    "wildfire_material_class",
    "sdes_wildfire_raw_0to5",
    "V_des_wildfire",
    "Risk_wildfire",
    "sdes_wildfire_note",
]
existing_show_cols = [c for c in show_cols if c in bridges_out.columns]
top20 = bridges_out.sort_values("Risk_wildfire", ascending=False).head(20)[existing_show_cols]
print("\nTop 20 by Risk_wildfire:")
print(top20.to_string(index=False))


✅ Done!
Wildfire years observed: 1992–2020  (T = 29 years)
Saved: wildfire.csv

Top 20 by Risk_wildfire:
STRUCTURE_NUMBER_008 coordinates (lat, lon)  N_wildfires_within_1mi  lambda_wildfire  p_wildfire  V_elem_wildfire  V_age_wildfire   wildfire_design_row_43B wildfire_material_class  sdes_wildfire_raw_0to5  V_des_wildfire  Risk_wildfire                                                      sdes_wildfire_note
     091610B00331001  31.561644, -97.128183                      78         2.689655    0.932096              1.0        3.600000              Truss - Thru                   Steel                     4.0        1.666667       5.592574                                                                        
     230250048002002  31.834242, -99.099828                      45         1.551724    0.788118              1.0        2.742857 Stringer/MultiBeam/Girder          Wood or Timber                     5.0        2.000000       4.323388                                               

### **hurricane risk**

In [ ]:
import numpy as np
import pandas as pd
from sklearn.neighbors import BallTree

# -----------------------
# USER SETTINGS
# -----------------------
BRIDGE_FILE  = "TX25.csv"
IBTRACS_FILE = "IBTrACS_TEXAS_ONLY.csv"
OUTPUT_FILE  = "texas_hurricane_risk_ranked.csv"

RADIUS_MILES        = 50.0
TIME_HORIZON_YEARS  = 1.0
CURRENT_YEAR        = 2026

# Texas sanity bounds (prevents Africa/ocean)
TX_LAT_MIN, TX_LAT_MAX = 25.0, 37.5
TX_LON_MIN, TX_LON_MAX = -107.5, -93.0

# -----------------------
# Helpers: Correct NBI lat/long conversion
# -----------------------
def nbi_lat_to_dd(x):
    """LAT_016: DDMMSSss -> decimal degrees. If already decimal (-90..90), keep."""
    if pd.isna(x):
        return np.nan
    try:
        v = float(x)
    except:
        return np.nan

    if -90.0 <= v <= 90.0:
        return v

    try:
        v = int(abs(v))
        d = v // 1_000_000
        m = (v // 10_000) % 100
        s = (v % 10_000) / 100.0
        return d + m / 60.0 + s / 3600.0
    except:
        return np.nan


def nbi_lon_to_dd(x):
    """LONG_017: DDDMMSSss -> decimal degrees (West negative). If already decimal, keep."""
    if pd.isna(x):
        return np.nan
    try:
        v = float(x)
    except:
        return np.nan

    if -180.0 <= v <= 180.0:
        return v if v <= 0 else -v

    try:
        v = int(abs(v))
        d = v // 1_000_000
        m = (v // 10_000) % 100
        s = (v % 10_000) / 100.0
        lon = d + m / 60.0 + s / 3600.0
        return -lon
    except:
        return np.nan


def miles_to_radians(miles):
    return miles / 3958.7613  # Earth radius in miles


def format_coord(lat, lon, decimals=6):
    if pd.isna(lat) or pd.isna(lon):
        return ""
    return f"{lat:.{decimals}f}, {lon:.{decimals}f}"


# -----------------------
# Damage level mapping
# -----------------------
def category_to_damage_level(cat):
    try:
        cat = int(cat)
    except:
        return "Unknown"

    mapping = {
        0: "No Risk",
        1: "Low",
        2: "Moderate",
        3: "High",
        4: "Very High",
        5: "Severe"
    }
    return mapping.get(cat, "Unknown")


# -----------------------
# SSHWS CATEGORY
# -----------------------
CAT_THRESH = {
    1: 74.0,
    2: 96.0,
    3: 111.0,
    4: 130.0,
    5: 157.0
}

def wind_to_cat(mph):
    if mph >= CAT_THRESH[5]:
        return 5
    if mph >= CAT_THRESH[4]:
        return 4
    if mph >= CAT_THRESH[3]:
        return 3
    if mph >= CAT_THRESH[2]:
        return 2
    if mph >= CAT_THRESH[1]:
        return 1
    return 0


# -----------------------
# 1) LOAD FILES
# -----------------------
bridges = pd.read_csv(BRIDGE_FILE, low_memory=False, dtype={"STRUCTURE_NUMBER_008": str})
storms  = pd.read_csv(IBTRACS_FILE, low_memory=False)

# -----------------------
# 2) BRIDGE COORDINATES (FIXED) + TEXAS SANITY FILTER
# -----------------------
if {"LAT_016", "LONG_017"}.issubset(bridges.columns):
    bridges["lat"] = bridges["LAT_016"].apply(nbi_lat_to_dd)
    bridges["lon"] = bridges["LONG_017"].apply(nbi_lon_to_dd)
elif {"LATITUDE", "LONGITUDE"}.issubset(bridges.columns):
    bridges["lat"] = pd.to_numeric(bridges["LATITUDE"], errors="coerce")
    bridges["lon"] = pd.to_numeric(bridges["LONGITUDE"], errors="coerce")
else:
    raise ValueError("Bridge coords not found. Expected LAT_016/LONG_017 or LATITUDE/LONGITUDE.")

# Texas sanity box
tx_ok = (
    bridges["lat"].between(TX_LAT_MIN, TX_LAT_MAX) &
    bridges["lon"].between(TX_LON_MIN, TX_LON_MAX)
)
bridges.loc[~tx_ok, ["lat", "lon"]] = np.nan

bridges["coordinates (lat, lon)"] = [
    format_coord(a, b, 6) for a, b in zip(bridges["lat"], bridges["lon"])
]

# Drop bridges without valid TX coords
bridges = bridges.dropna(subset=["lat", "lon"]).copy()
bridges.reset_index(drop=True, inplace=True)

print(f"Bridges with valid TX coordinates: {len(bridges):,}")

# -----------------------
# 3) PREP STORM DATA (IBTrACS)
# -----------------------
storms = storms.rename(columns={
    "Storm Identifier": "storm_id",
    "Calendar Year": "year",
    "Geograpic Latitude": "lat",
    "Geographic Longitude": "lon",
    "Maximum Sustained Wind (knots)": "wind_kt",
})

required_storm_cols = ["storm_id", "year", "lat", "lon", "wind_kt"]
for col in required_storm_cols:
    if col not in storms.columns:
        raise ValueError(f"Missing required storm column: {col}")

storms["lat"] = pd.to_numeric(storms["lat"], errors="coerce")
storms["lon"] = pd.to_numeric(storms["lon"], errors="coerce")
storms["wind_kt"] = pd.to_numeric(storms["wind_kt"], errors="coerce")
storms["year"] = pd.to_numeric(storms["year"], errors="coerce")

storms = storms.dropna(subset=["storm_id", "year", "lat", "lon", "wind_kt"]).copy()

# Wind mph
storms["wind_mph"] = storms["wind_kt"] * 1.15078

# Texas filter for storm points too
storms = storms[
    storms["lat"].between(TX_LAT_MIN, TX_LAT_MAX) &
    storms["lon"].between(TX_LON_MIN, TX_LON_MAX)
].copy()

storms.reset_index(drop=True, inplace=True)

if len(storms) == 0:
    raise ValueError("No valid Texas storm points found after cleaning/filtering.")

min_year = int(storms["year"].min())
max_year = int(storms["year"].max())
years_covered = max(1, (max_year - min_year + 1))
print(f"IBTrACS years covered: {min_year}–{max_year} (Y={years_covered})")

# -----------------------
# 4) STORM CATEGORY
# -----------------------
storms["cat"] = storms["wind_mph"].apply(wind_to_cat)

# -----------------------
# 5) BALLTREE FOR STORMS
# -----------------------
storm_points_deg = storms[["lat", "lon"]].to_numpy(dtype=float)
storm_points_rad = np.radians(storm_points_deg)

tree = BallTree(storm_points_rad, metric="haversine")
radius_rad = miles_to_radians(RADIUS_MILES)

storm_id_arr   = storms["storm_id"].to_numpy()
storm_cat_arr  = storms["cat"].to_numpy()
storm_wkt_arr  = storms["wind_kt"].to_numpy()
storm_wmph_arr = storms["wind_mph"].to_numpy()

# -----------------------
# 6) HAZARD COUNTS + PROBABILITY + BRIDGE CATEGORY
# -----------------------
bridge_points_deg = bridges[["lat", "lon"]].to_numpy(dtype=float)
bridge_points_rad = np.radians(bridge_points_deg)
neighbors = tree.query_radius(bridge_points_rad, r=radius_rad)

N_by_cat = {g: np.zeros(len(bridges), dtype=int) for g in range(1, 6)}
p_by_cat = {g: np.zeros(len(bridges), dtype=float) for g in range(1, 6)}

bridge_hurricane_category = np.zeros(len(bridges), dtype=int)
max_sustained_wind_kt_nearby = np.zeros(len(bridges), dtype=float)
max_sustained_wind_mph_nearby = np.zeros(len(bridges), dtype=float)

for i, idxs in enumerate(neighbors):
    if idxs.size == 0:
        continue

    sub_ids   = storm_id_arr[idxs]
    sub_cats  = storm_cat_arr[idxs]
    sub_wkt   = storm_wkt_arr[idxs]
    sub_wmph  = storm_wmph_arr[idxs]

    # max nearby sustained wind
    max_sustained_wind_kt_nearby[i] = np.nanmax(sub_wkt) if len(sub_wkt) else 0.0
    max_sustained_wind_mph_nearby[i] = np.nanmax(sub_wmph) if len(sub_wmph) else 0.0

    # one max category per storm
    maxcat = {}
    for sid, c in zip(sub_ids, sub_cats):
        if sid not in maxcat or c > maxcat[sid]:
            maxcat[sid] = c

    maxcats = np.array(list(maxcat.values()), dtype=int)

    # final bridge-level hurricane category
    bridge_hurricane_category[i] = int(maxcats.max()) if len(maxcats) > 0 else 0

    # cumulative counts/probabilities
    for g in range(1, 6):
        n = int(np.sum(maxcats >= g))
        N_by_cat[g][i] = n
        lam = n / years_covered
        p_by_cat[g][i] = 1.0 - np.exp(-lam * TIME_HORIZON_YEARS)

for g in range(1, 6):
    bridges[f"N_storm_cat{g}"] = N_by_cat[g]
    bridges[f"p_cat{g}"] = p_by_cat[g]

bridges["bridge_hurricane_category"] = bridge_hurricane_category
bridges["damage_level"] = bridges["bridge_hurricane_category"].apply(category_to_damage_level)
bridges["max_sustained_wind_kt_nearby"] = max_sustained_wind_kt_nearby
bridges["max_sustained_wind_mph_nearby"] = max_sustained_wind_mph_nearby

# -----------------------
# 7) VULNERABILITY MULTIPLIERS (NBI-based proxies)
# -----------------------
def v_des_from_43a(x):
    try:
        x = int(float(x))
    except:
        return 1.0

    if x in [1, 2]:       # concrete
        s = 2
    elif x in [3, 4]:     # steel
        s = 3
    elif x == 5:          # timber
        s = 5
    else:
        s = 3

    return (s + 1.0) / 3.0


if "STRUCTURE_KIND_043A" in bridges.columns:
    bridges["V_des_hurr"] = bridges["STRUCTURE_KIND_043A"].apply(v_des_from_43a)
else:
    bridges["V_des_hurr"] = 1.0


def v_scour_from_113(code):
    if pd.isna(code):
        return 1.0
    c = str(code).strip().upper()
    high_bin = {"0", "1", "2", "3", "4", "6", "U"}
    low_bin  = {"5", "7", "8", "9", "N", "T"}

    if c in high_bin:
        return 1.2
    if c in low_bin:
        return 0.4
    return 1.0


scour_col = None
for c in ["SCOUR_CRITICAL_113", "SCOUR_CRITICAL_113A", "SCOUR_113", "NBI_113"]:
    if c in bridges.columns:
        scour_col = c
        break

if scour_col is None:
    cand = [c for c in bridges.columns if "113" in c.upper()]
    scour_col = cand[0] if cand else None

bridges["V_scour_hurr"] = bridges[scour_col].apply(v_scour_from_113) if scour_col else 1.0

BASE_YEAR = 1990
DENOM = 35.0

def v_age(year_built):
    y = pd.to_numeric(year_built, errors="coerce")
    if pd.isna(y):
        return 1.0
    return max(1.0, 1.0 + (BASE_YEAR - float(y)) / DENOM)


if "YEAR_BUILT_027" in bridges.columns:
    bridges["V_age_hurr"] = bridges["YEAR_BUILT_027"].apply(v_age)
else:
    bridges["V_age_hurr"] = 1.0


cond_cols = [
    c for c in ["DECK_COND_058", "SUPERSTRUCTURE_COND_059", "SUBSTRUCTURE_COND_060"]
    if c in bridges.columns
]

def rating_to_s0(r):
    r = pd.to_numeric(r, errors="coerce")
    if pd.isna(r):
        return 2.5
    s = (9.0 - float(r)) * (5.0 / 9.0)
    return float(np.clip(s, 0.0, 5.0))


if cond_cols:
    base_s = bridges[cond_cols].apply(lambda col: col.map(rating_to_s0)).mean(axis=1)
else:
    base_s = pd.Series(np.full(len(bridges), 2.5), index=bridges.index)

for g in range(1, 6):
    sg = np.clip(base_s + 0.5 * (g - 1), 0.0, 5.0)
    bridges[f"V_elem_cat{g}"] = (sg + 1.0) / 3.0

# -----------------------
# 8) RISK INDEX
# -----------------------
bridges["Risk_hurricane"] = 0.0

for g in range(1, 6):
    bridges["Risk_hurricane"] += (
        bridges[f"p_cat{g}"] *
        bridges[f"V_elem_cat{g}"] *
        bridges["V_des_hurr"] *
        bridges["V_scour_hurr"] *
        bridges["V_age_hurr"]
    )

# -----------------------
# 9) SAVE FINAL CSV
# -----------------------
required_cols = [
    "STRUCTURE_NUMBER_008",
    "LOCATION_009",
    "lat",
    "lon",
    "coordinates (lat, lon)",
    "YEAR_BUILT_027",
    "bridge_hurricane_category",
    "damage_level",
    "max_sustained_wind_kt_nearby",
    "max_sustained_wind_mph_nearby",
    "V_des_hurr",
    "V_scour_hurr",
    "V_age_hurr",
    "Risk_hurricane",
]

required_cols = [c for c in required_cols if c in bridges.columns]

out = bridges[required_cols].copy().sort_values("Risk_hurricane", ascending=False)
out.to_csv(OUTPUT_FILE, index=False)

print(f"\n✅ Saved ranked results: {OUTPUT_FILE}")
print(out.head(10).to_string(index=False))

blank_coords = (
    (out["coordinates (lat, lon)"].astype(str).str.len() == 0).sum()
    if "coordinates (lat, lon)" in out.columns else 0
)
print(f"\nBridges in output: {len(out):,} | Blank coordinate strings: {blank_coords:,}")

Bridges with valid TX coordinates: 56,951
IBTrACS years covered: 1993–2024 (Y=32)

✅ Saved ranked results: texas_hurricane_risk_ranked.csv
STRUCTURE_NUMBER_008               LOCATION_009       lat        lon coordinates (lat, lon)  YEAR_BUILT_027  bridge_hurricane_category damage_level  max_sustained_wind_kt_nearby  max_sustained_wind_mph_nearby  V_des_hurr  V_scour_hurr  V_age_hurr  Risk_hurricane
     201240036802018       '1.9 MI NE OF SH 73' 29.843211 -94.330889  29.843211, -94.330889            1937                          3         High                         100.0                       115.0780    1.000000           1.2    2.514286        0.697978
     201240AA0275002       '1.4 MI S OF SH 124' 30.002308 -94.155094  30.002308, -94.155094            1970                          3         High                         100.0                       115.0780    2.000000           1.2    1.571429        0.683017
     161960018002018       '3.8 MI N OF FM 774' 28.376775 -96.914556  28

### **Explosive risk**

In [ ]:
import pandas as pd
import numpy as np

# =========================
# FILE PATHS
# =========================
IN_CSV  = "tx25-final.csv"
OUT_CSV = "explosive_risk_ranked.csv"

# =========================
# LOAD DATA
# =========================
df = pd.read_csv(IN_CSV, low_memory=False)

# =========================
# REQUIRED NBI COLUMNS
# =========================
REQ_COLS = [
    "STRUCTURE_NUMBER_008",
    "ADT_029",
    "DETOUR_KILOS_019",
    "HISTORY_037",
    "LAT_016",
    "LONG_017",
]

missing = [c for c in REQ_COLS if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

# =========================
# CLEAN / TYPE CAST
# =========================
df["ADT_029"] = pd.to_numeric(df["ADT_029"], errors="coerce")
df["DETOUR_KILOS_019"] = pd.to_numeric(df["DETOUR_KILOS_019"], errors="coerce")
df["HISTORY_037"] = pd.to_numeric(df["HISTORY_037"], errors="coerce")
df["LAT_016"] = pd.to_numeric(df["LAT_016"], errors="coerce")
df["LONG_017"] = pd.to_numeric(df["LONG_017"], errors="coerce")

# =========================
# FIXED NBI COORDINATE CONVERSION
# =========================
def nbi_lat_to_dd(x):
    if pd.isna(x):
        return np.nan
    try:
        v = int(abs(x))
        d = v // 1_000_000
        m = (v // 10_000) % 100
        s = (v % 10_000) / 100.0
        return d + m / 60.0 + s / 3600.0
    except:
        return np.nan

def nbi_lon_to_dd(x):
    if pd.isna(x):
        return np.nan
    try:
        v = int(abs(x))
        d = v // 1_000_000
        m = (v // 10_000) % 100
        s = (v % 10_000) / 100.0
        lon = d + m / 60.0 + s / 3600.0
        return -lon   # force west negative
    except:
        return np.nan

df["lat"] = df["LAT_016"].apply(nbi_lat_to_dd)
df["lon"] = df["LONG_017"].apply(nbi_lon_to_dd)

# =========================
# TEXAS SANITY FILTER (critical)
# =========================
TX_LAT_MIN, TX_LAT_MAX = 25.0, 37.5
TX_LON_MIN, TX_LON_MAX = -107.5, -93.0

valid_tx = (
    df["lat"].between(TX_LAT_MIN, TX_LAT_MAX) &
    df["lon"].between(TX_LON_MIN, TX_LON_MAX)
)

df.loc[~valid_tx, ["lat", "lon"]] = np.nan

# =========================
# COORDINATES COLUMN (correct)
# =========================
def make_coord(lat, lon, ndigits=6):
    if pd.isna(lat) or pd.isna(lon):
        return np.nan
    return f"({round(lat, ndigits)}, {round(lon, ndigits)})"

df["COORDINATES"] = df.apply(lambda r: make_coord(r["lat"], r["lon"]), axis=1)

# =========================
# STEP 1: SYMBOLIC IMPORTANCE (S)
# =========================
S_MAP = {1: 6, 2: 5, 3: 4, 4: 3, 5: 1}
df["S_symbolic"] = df["HISTORY_037"].map(S_MAP).fillna(2)

# =========================
# STEP 2: ECONOMIC IMPORTANCE (E)
# =========================
def map_E(adt):
    if pd.isna(adt): return 3
    if adt < 100: return 1
    if adt < 5_000: return 2
    if adt < 25_000: return 3
    if adt < 100_000: return 4
    if adt < 200_000: return 5
    return 6

df["E_economic"] = df["ADT_029"].apply(map_E)

# =========================
# STEP 3: DETOUR IMPACT (D)
# =========================
df["DETOUR_MILES"] = df["DETOUR_KILOS_019"] * 0.621371

def map_D(miles):
    if pd.isna(miles): return 3
    if miles <= 10: return 2
    if miles <= 20: return 4
    if miles <= 50: return 5
    if miles <= 100: return 6
    return 6

df["D_detour"] = df["DETOUR_MILES"].apply(map_D)

# =========================
# STEP 4: CRITICALITY FACTOR
# =========================
df["ED"] = df["E_economic"] * df["D_detour"]
df["CF"] = df["S_symbolic"] + df["ED"]

# =========================
# OUTPUT (Ranked)
# =========================
out_cols = [
    "STRUCTURE_NUMBER_008",
    "LOCATION_009",
    "COORDINATES",
    "lat",
    "lon",
    "ADT_029",
    "DETOUR_KILOS_019",
    "DETOUR_MILES",
    "HISTORY_037",
    "S_symbolic",
    "E_economic",
    "D_detour",
    "ED",
    "CF",
]

ranked = df[out_cols].sort_values(
    by=["CF", "ED", "E_economic"],
    ascending=[False, False, False],
    kind="mergesort"
)

ranked.to_csv(OUT_CSV, index=False)

print("✅ Criticality assessment completed with CORRECT Texas locations")
print(f"📁 Saved to: {OUT_CSV}")
print("Valid Texas coordinate rows:", ranked["lat"].notna().sum())


✅ Criticality assessment completed with CORRECT Texas locations
📁 Saved to: explosive_risk_ranked.csv
Valid Texas coordinate rows: 56951


### **Vessel Collision Risk**

In [ ]:
import pandas as pd
import numpy as np

# ============================================================
# FDOT Vessel-Collision Risk (NBI-only screening)
# - Uses NBI Item 111 (Pier Protection for Navigation)
# - Uses Year Built (Item 27) for age vulnerability
# - Element vulnerability forced to 1.0 (assumption)
# - FIXED: NBI packed coordinates -> correct decimal degrees
# - OUTPUT: only the exact columns you requested (in exact order)
# ============================================================

INPUT_CSV = "TX-25.csv"
OUT_ALL   = "risk_vessel.csv"

# FDOT PLAT baseline annual probability (0.381%)
P0 = 0.00381

# Texas sanity bounds (removes Africa/ocean outliers caused by bad coords)
TX_LAT_MIN, TX_LAT_MAX = 25.0, 37.5
TX_LON_MIN, TX_LON_MAX = -107.5, -93.0

# -------------------------
# Helpers: NBI coord conversion
# -------------------------
def nbi_lat_to_dd(x):
    """LAT_016: DDMMSSss -> decimal degrees. If already decimal (-90..90), keep."""
    if pd.isna(x):
        return np.nan
    try:
        v = float(x)
    except:
        return np.nan

    # already decimal latitude
    if -90.0 <= v <= 90.0:
        return v

    try:
        v = int(abs(v))
        d = v // 1_000_000
        m = (v // 10_000) % 100
        s = (v % 10_000) / 100.0
        return d + m / 60.0 + s / 3600.0
    except:
        return np.nan

def nbi_lon_to_dd(x):
    """LONG_017: DDDMMSSss -> decimal degrees (West negative). If already decimal, keep."""
    if pd.isna(x):
        return np.nan
    try:
        v = float(x)
    except:
        return np.nan

    # already decimal longitude
    if -180.0 <= v <= 180.0:
        return v if v <= 0 else -v

    try:
        v = int(abs(v))
        d = v // 1_000_000
        m = (v // 10_000) % 100
        s = (v % 10_000) / 100.0
        lon = d + m / 60.0 + s / 3600.0
        return -lon
    except:
        return np.nan

def format_coord(lat, lon, decimals=6):
    if pd.isna(lat) or pd.isna(lon):
        return ""
    return f"{lat:.{decimals}f}, {lon:.{decimals}f}"

def pick_first_existing(df, candidates):
    return next((c for c in candidates if c in df.columns), None)

# -------------------------
# 1) Load
# -------------------------
df = pd.read_csv(INPUT_CSV, low_memory=False, dtype={"STRUCTURE_NUMBER_008": str})

# -------------------------
# 2) Coordinates (FIXED)
# -------------------------
lat_col = pick_first_existing(df, ["LAT_016", "LAT_016A", "LATITUDE_016", "LATITUDE", "LAT"])
lon_col = pick_first_existing(df, ["LONG_017", "LON_017", "LONG_017A", "LONGITUDE_017", "LONGITUDE", "LON", "LONG"])

if lat_col is None or lon_col is None:
    df["coordinates (lat, lon)"] = ""
else:
    df["_lat_dd"] = pd.to_numeric(df[lat_col], errors="coerce").apply(nbi_lat_to_dd)
    df["_lon_dd"] = pd.to_numeric(df[lon_col], errors="coerce").apply(nbi_lon_to_dd)

    # Texas sanity filter to kill Africa/ocean outliers
    ok = df["_lat_dd"].between(TX_LAT_MIN, TX_LAT_MAX) & df["_lon_dd"].between(TX_LON_MIN, TX_LON_MAX)
    df.loc[~ok, ["_lat_dd", "_lon_dd"]] = np.nan

    df["coordinates (lat, lon)"] = [
        format_coord(a, b, 6) for a, b in zip(df["_lat_dd"], df["_lon_dd"])
    ]

# -------------------------
# 3) Find Item 111 column
# -------------------------
item111_candidates = [
    "PIER_PROTECTION_111",
    "PIER_PROT_111",
    "PIER_PROTECTION_111A",
    "NAV_PROTECTION_111",
    "PIER_OR_ABUTMENT_PROTECTION_111",
    "NBI_111",
    "ITEM_111",
    "PIER_ABUTMENT_PROTECTION_111",
]
item111_col = pick_first_existing(df, item111_candidates)

# fallback search
if item111_col is None:
    for c in df.columns:
        cu = c.upper()
        if ("111" in cu) and (("PROT" in cu) or ("PIER" in cu) or ("NAV" in cu)):
            item111_col = c
            break

if item111_col is None:
    raise ValueError("Could not find NBI Item 111 column in TX25.csv (pier protection).")

# Normalize Item 111 values
raw111 = df[item111_col].astype(str).str.strip().str.upper()
raw111 = raw111.replace({"": np.nan, "NAN": np.nan, "NONE": np.nan})

# numeric codes; keep 'N' as special (non-numeric)
i111_num = pd.to_numeric(raw111.where(~raw111.isin(["N"]), np.nan), errors="coerce")

# Exposed bridges: 2,3,4,5
exposed = i111_num.isin([2, 3, 4, 5])

# -------------------------
# 4) pvessel
# -------------------------
df["pvessel"] = np.where(exposed, P0, 0.0)

# -------------------------
# 5) sdes + Vdes from Item 111
# -------------------------
sdes_map = {1: 0, 2: 2, 3: 3, 4: 4, 5: 5}
df["sdes_vessel"] = i111_num.map(sdes_map)

df["V_des_vessel"] = (df["sdes_vessel"] + 1.0) / 3.0
df["V_des_vessel"] = df["V_des_vessel"].fillna(1.0)

# -------------------------
# 6) V_age from Year Built (Item 27)
# -------------------------
year_col = pick_first_existing(df, ["YEAR_BUILT_027", "YEAR_BUILT", "YR_BUILT_027", "NBI_027", "ITEM_027"])
if year_col is None:
    for c in df.columns:
        cu = c.upper()
        if ("YEAR" in cu) and ("BUILT" in cu):
            year_col = c
            break

if year_col is None:
    df["V_age_vessel"] = 1.0
else:
    Y = pd.to_numeric(df[year_col], errors="coerce")
    df["V_age_vessel"] = 1.0 + (1985.0 - Y) / 30.0
    df["V_age_vessel"] = df["V_age_vessel"].clip(lower=1.0).fillna(1.0)

# -------------------------
# 7) V_elem assumed 1
# -------------------------
df["V_elem_vessel"] = 1.0

# -------------------------
# 8) Risk index
# -------------------------
df["Risk_vessel"] = df["pvessel"] * df["V_elem_vessel"] * df["V_age_vessel"] * df["V_des_vessel"]

# -------------------------
# 9) Rank
# -------------------------
df_ranked = df.sort_values("Risk_vessel", ascending=False, kind="mergesort").reset_index(drop=True)

# -------------------------
# 10) Make output EXACT columns & names requested
#     (Rename found columns to the exact expected output names.)
# -------------------------
# Ensure LOCATION_009 exists (blank if missing)
if "LOCATION_009" not in df_ranked.columns:
    df_ranked["LOCATION_009"] = ""

# Rename the found item111/year columns to exact names requested
if item111_col != "PIER_PROTECTION_111":
    df_ranked["PIER_PROTECTION_111"] = df_ranked[item111_col]
else:
    # already correct name
    pass

if year_col and (year_col != "YEAR_BUILT_027"):
    df_ranked["YEAR_BUILT_027"] = df_ranked[year_col]
elif not year_col:
    df_ranked["YEAR_BUILT_027"] = np.nan

FINAL_COLS = [
    "STRUCTURE_NUMBER_008",
    "LOCATION_009",
    "coordinates (lat, lon)",
    "PIER_PROTECTION_111",
    "YEAR_BUILT_027",
    "pvessel",
    "sdes_vessel",
    "V_des_vessel",
    "V_age_vessel",
    "V_elem_vessel",
    "Risk_vessel",
]

# If any are missing for some reason, create them (stable export)
for c in FINAL_COLS:
    if c not in df_ranked.columns:
        df_ranked[c] = np.nan

df_out = df_ranked[FINAL_COLS].copy()
top25  = df_out.head(25).copy()

df_out.to_csv(OUT_ALL, index=False)
top25.to_csv(OUT_TOP25, index=False)

print("✅ Done.")
print(f"Item 111 source column used: {item111_col} -> exported as PIER_PROTECTION_111")
print(f"Year built source column used: {year_col} -> exported as YEAR_BUILT_027")
print(f"Saved ranked: {OUT_ALL}")
print(f"Saved top 25: {OUT_TOP25}")
print("\nTop 10 preview:")
print(df_out.head(10).to_string(index=False))


✅ Done.
Item 111 source column used: PIER_PROTECTION_111 -> exported as PIER_PROTECTION_111
Year built source column used: YEAR_BUILT_027 -> exported as YEAR_BUILT_027
Saved ranked: risk_vessel.csv
Saved top 25: top_25_flood_scour_risk.csv

Top 10 preview:
STRUCTURE_NUMBER_008                LOCATION_009 coordinates (lat, lon)  PIER_PROTECTION_111  YEAR_BUILT_027  pvessel  sdes_vessel  V_des_vessel  V_age_vessel  V_elem_vessel  Risk_vessel
     201010006506079        '2.4 MI S OF SH 105'  30.179247, -94.186139                  5.0            1953  0.00381          5.0      2.000000      2.066667            1.0     0.015748
     161780010106041 '3.16 M SW OF NUECES CO LN'  27.812728, -97.395339                  5.0            1959  0.00381          5.0      2.000000      1.866667            1.0     0.014224
     121020AA9962004        '2.40 MI SW OF I-10'  29.762075, -95.080486                  2.0            1930  0.00381          2.0      1.000000      2.833333            1.0     0.01

### **Advanced deterioration risk**

In [ ]:
# ============================================================
# FDOT Advanced Deterioration Risk (NBI-Proxy Implementation)
# FIXED: Correct NBI 43A material mapping (code 7 = Timber, not 6)
#
# Input : TX25.csv
# Output: deteriorationn_risk.csv (sorted high->low)
# ============================================================

import pandas as pd
import numpy as np
from scipy.stats import norm

# -----------------------------
# CONFIG
# -----------------------------
INPUT_CSV = "TX25.csv"
OUT_CSV   = "deteriorationn_risk.csv"

TX_LAT_MIN, TX_LAT_MAX = 25.0, 37.5
TX_LON_MIN, TX_LON_MAX = -107.5, -93.0

# -----------------------------
# FDOT TABLES
# -----------------------------
HAZARD_PARAMS = {
    "Concrete – prestressed": dict(a=3000, b=0.00199, c=0.0, mu=15.461, sigma=3.982),
    "Concrete – reinforced":  dict(a=3000, b=0.00047, c=0.0, mu=15.385, sigma=3.928),
    "Steel":                  dict(a=3000, b=0.00196, c=0.0, mu=15.545, sigma=3.998),
    "Timber":                 dict(a=3000, b=0.00539, c=0.0, mu=15.077, sigma=3.902),
}

DECAY_COEFFS = {
    "Concrete – prestressed": dict(W_deck=0.20, W_super=0.40, W_sub=0.40, wc2_deck=0.50, wc2_super=0.50, wc2_sub=0.50),
    "Concrete – reinforced":  dict(W_deck=0.20, W_super=0.40, W_sub=0.40, wc2_deck=0.50, wc2_super=0.50, wc2_sub=0.50),
    "Steel":                  dict(W_deck=0.20, W_super=0.40, W_sub=0.40, wc2_deck=0.50, wc2_super=0.50, wc2_sub=0.50),
    "Timber":                 dict(W_deck=0.40, W_super=0.40, W_sub=0.20, wc2_deck=0.10, wc2_super=0.50, wc2_sub=0.50),
}

# -----------------------------
# HELPERS
# -----------------------------
def map_43A_to_material(code):
    """
    Map NBI Item 43A codes to FDOT material groups.
    Source: NBI Coding Guide — Items 43A, 44A
      1 = Concrete
      2 = Concrete continuous
      3 = Steel
      4 = Steel continuous
      5 = Prestressed concrete
      6 = Prestressed concrete continuous
      7 = Wood or timber
      8 = Masonry
      9 = Aluminum, Wrought Iron or Cast Iron
      0 = Other
    """
    try:
        if pd.isna(code):
            return "Concrete – reinforced"
        code = int(code)
    except:
        return "Concrete – reinforced"

    if code in (3, 4):
        return "Steel"
    if code == 7:
        return "Timber"                   # 7 = Wood or Timber
    if code in (5, 6):
        return "Concrete – prestressed"   # 5 = PSC, 6 = PSC Continuous
    if code in (1, 2):
        return "Concrete – reinforced"    # 1 = Concrete, 2 = Concrete Continuous
    return "Concrete – reinforced"        # 0, 8, 9 default

def rating_to_severity_fraction(r):
    """Proxy severity fraction in [0,1] from NBI rating 0..9 (9 best)."""
    if pd.isna(r):
        return np.nan
    try:
        r = float(r)
    except:
        return np.nan

    r = max(0.0, min(9.0, r))
    mapping = {
        9: 0.01, 8: 0.03, 7: 0.07, 6: 0.15, 5: 0.30,
        4: 0.50, 3: 0.70, 2: 0.85, 1: 0.95, 0: 0.98
    }
    return mapping.get(int(round(r)), 0.30)

def compute_p_advdet(D, params):
    """
    FDOT Eq (1.8):
      p = a * [phi(z) / (sigma * D * (1 - Phi(z)))] + b*D + c
      z = (ln D - mu) / sigma
    """
    if (D is None) or pd.isna(D) or D <= 0:
        return 0.0

    a, b, c, mu, sigma = params["a"], params["b"], params["c"], params["mu"], params["sigma"]
    lnD = np.log(D)
    z = (lnD - mu) / sigma

    phi = norm.pdf(z)
    Phi = norm.cdf(z)

    denom = sigma * D * (1.0 - Phi)
    term1 = (a * phi / denom) if denom > 0 else 0.0
    term2 = b * D + c
    p = term1 + term2

    if not np.isfinite(p):
        p = 0.0
    return float(np.clip(p, 0.0, 1.0))

def nbi_lat_to_dd(x):
    if pd.isna(x): return np.nan
    try:
        v = float(x)
    except:
        return np.nan
    if -90.0 <= v <= 90.0:
        return v
    try:
        v = int(abs(v))
        d = v // 1_000_000
        m = (v // 10_000) % 100
        s = (v % 10_000) / 100.0
        return d + m/60.0 + s/3600.0
    except:
        return np.nan

def nbi_lon_to_dd(x):
    if pd.isna(x): return np.nan
    try:
        v = float(x)
    except:
        return np.nan
    if -180.0 <= v <= 180.0:
        return v if v <= 0 else -v
    try:
        v = int(abs(v))
        d = v // 1_000_000
        m = (v // 10_000) % 100
        s = (v % 10_000) / 100.0
        lon = d + m/60.0 + s/3600.0
        return -lon
    except:
        return np.nan

def format_coord(lat, lon, decimals=6):
    if pd.isna(lat) or pd.isna(lon):
        return ""
    return f"{lat:.{decimals}f}, {lon:.{decimals}f}"

# -----------------------------
# MAIN
# -----------------------------
df = pd.read_csv(INPUT_CSV, low_memory=False, dtype={"STRUCTURE_NUMBER_008": str})

need_cols = [
    "STRUCTURE_NUMBER_008", "LOCATION_009",
    "DECK_COND_058", "SUPERSTRUCTURE_COND_059", "SUBSTRUCTURE_COND_060",
    "STRUCTURE_KIND_043A", "LAT_016", "LONG_017",
]
missing = [c for c in need_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns in TX25.csv: {missing}")

for c in ["DECK_COND_058", "SUPERSTRUCTURE_COND_059", "SUBSTRUCTURE_COND_060",
          "STRUCTURE_KIND_043A", "LAT_016", "LONG_017"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# Coordinates
df["_lat_dd"] = df["LAT_016"].apply(nbi_lat_to_dd)
df["_lon_dd"] = df["LONG_017"].apply(nbi_lon_to_dd)

ok = df["_lat_dd"].between(TX_LAT_MIN, TX_LAT_MAX) & df["_lon_dd"].between(TX_LON_MIN, TX_LON_MAX)
df.loc[~ok, ["_lat_dd", "_lon_dd"]] = np.nan

df["coordinates (lat, lon)"] = [
    format_coord(a, b, 6) for a, b in zip(df["_lat_dd"], df["_lon_dd"])
]

# Material group — FIXED mapping
df["MAT_GROUP"] = df["STRUCTURE_KIND_043A"].apply(map_43A_to_material)

# Severity fractions
df["F_deck"]  = df["DECK_COND_058"].apply(rating_to_severity_fraction)
df["F_super"] = df["SUPERSTRUCTURE_COND_059"].apply(rating_to_severity_fraction)
df["F_sub"]   = df["SUBSTRUCTURE_COND_060"].apply(rating_to_severity_fraction)
df[["F_deck", "F_super", "F_sub"]] = df[["F_deck", "F_super", "F_sub"]].fillna(0.30)

# Decay index
def compute_D(row):
    coeff = DECAY_COEFFS[row["MAT_GROUP"]]
    D = 100.0 * (
        coeff["W_deck"]  * row["F_deck"] +
        coeff["W_super"] * row["F_super"] +
        coeff["W_sub"]   * row["F_sub"]
    )
    return float(np.clip(D, 0.0, 100.0))

df["D_decay_index"] = df.apply(compute_D, axis=1)

# p_advdet
def compute_p(row):
    params = HAZARD_PARAMS[row["MAT_GROUP"]]
    return compute_p_advdet(row["D_decay_index"], params)

df["p_advdet"]   = df.apply(compute_p, axis=1)
df["Risk_advdet"] = df["p_advdet"]

# Add Risk_Level
df_sorted = df.sort_values("Risk_advdet", ascending=False).reset_index(drop=True)
df_sorted["Risk_Rank"] = df_sorted.index + 1
df_sorted["Risk_Level"] = df_sorted["Risk_Rank"].apply(
    lambda r: "Top 20" if r <= 20 else "Top 50" if r <= 50 else "Top 100" if r <= 100 else "All Bridges"
)

# Final output
FINAL_COLS = [
    "STRUCTURE_NUMBER_008", "LOCATION_009", "MAT_GROUP",
    "DECK_COND_058", "SUPERSTRUCTURE_COND_059", "SUBSTRUCTURE_COND_060",
    "F_deck", "F_super", "F_sub",
    "D_decay_index", "p_advdet", "Risk_advdet",
    "coordinates (lat, lon)", "Risk_Level",
]

ranked = df_sorted[FINAL_COLS]
ranked.to_csv(OUT_CSV, index=False)

print(f"Done. Saved to: {OUT_CSV}")
print(f"Rows: {len(ranked):,}")
print("\nMAT_GROUP distribution:")
print(ranked["MAT_GROUP"].value_counts())
print("\nTop 10:")
print(ranked.head(10)[["LOCATION_009","MAT_GROUP","D_decay_index","Risk_advdet"]].to_string())

Done. Saved to: deteriorationn_risk.csv
Rows: 56,951

MAT_GROUP distribution:
MAT_GROUP
Concrete – reinforced     30361
Concrete – prestressed    19491
Steel                      6812
Timber                      287
Name: count, dtype: int64

Top 10:
                 LOCATION_009 MAT_GROUP  D_decay_index  Risk_advdet
0      '0.20 MI S  OF FM 777'    Timber           76.0     0.500884
1        '0.2 MI NW OF US 84'    Timber           71.0     0.475751
2       '0.35 MI NW OF CR 99'    Timber           50.0     0.372027
3     '1.40 MI W  OF FM 2626'    Timber           49.0     0.367186
4      '2.5 MI SE OF FM 1793'    Timber           46.0     0.352734
5  '0.4 MI SE COTTONWOOD SCH'    Timber           43.0     0.338400
6      '1.1 MI E  OF FM 1804'    Timber           42.0     0.333650
7  '0.10 MI S  OF CLINTON DR'    Timber           42.0     0.333650
8       '1.60 MI SE OF SH 36'    Timber           41.0     0.328916
9        '0.95 MI SW OF SH 6'    Timber           38.0     0.314814


### **Tornado risk**

In [ ]:
# ============================================================
# Texas Tornado Risk (1950–2010) — FDOT-style simplified (FAST + CORRECT)
# Risk = p_tornado * V_age, with V_elem = 1 and V_des = 1
# Counts tornado touchdown points within 1 mile of each bridge.
# ============================================================

import re
import numpy as np
import pandas as pd
from sklearn.neighbors import BallTree

# -----------------------------
# 0) Paths + settings
# -----------------------------
BRIDGE_CSV  = "TX25.csv"
TORNADO_CSV = "1950-2024_all_tornadoes.csv"
OUT_CSV     = "tornadoo_risk_1950_2010.csv"

YEAR_START = 1950
YEAR_END   = 2010
T_YEARS    = (YEAR_END - YEAR_START + 1)  # 61 years inclusive

BUFFER_MILES = 1.0
EARTH_RADIUS_M = 6371008.8
RADIUS_M = BUFFER_MILES * 1609.344
RADIUS_RAD = RADIUS_M / EARTH_RADIUS_M

CLIP_VAGE_MIN_1 = True

# -----------------------------
# 1) Helpers
# -----------------------------
def pick_col(df: pd.DataFrame, patterns, required=True) -> str:
    cols = list(df.columns)
    for pat in patterns:
        rx = re.compile(pat, flags=re.IGNORECASE)
        for c in cols:
            if rx.search(c):
                return c
    if required:
        raise ValueError(f"Could not find a column matching patterns={patterns}. "
                         f"Some available columns: {cols[:80]}")
    return None

# --- REPLACE ONLY THIS FUNCTION in your code ---
def to_decimal_degrees(series: pd.Series, is_lon=False) -> pd.Series:
    """
    FIX for TX25: LAT_016 / LONG_017 are NBI packed coords, not microdegrees.
      LAT:  DDMMSSss  -> dd + mm/60 + (ss.ss)/3600
      LON:  DDDMMSSss -> -(ddd + mm/60 + (ss.ss)/3600)  (West negative)
    If values already look like decimal degrees, keep them.
    """
    s = pd.to_numeric(series, errors="coerce")

    out = pd.Series(np.nan, index=series.index, dtype="float64")

    # Case 1: already decimal degrees
    if is_lon:
        mask_dd = s.between(-180, 180)
        out.loc[mask_dd] = s.loc[mask_dd]
        # force west negative if someone stored lon as +95
        out.loc[mask_dd & (out > 0)] = -out.loc[mask_dd & (out > 0)]
    else:
        mask_dd = s.between(-90, 90)
        out.loc[mask_dd] = s.loc[mask_dd]

    # Case 2: NBI packed format
    mask_packed = out.isna() & s.notna()
    if mask_packed.any():
        v = s.loc[mask_packed].abs().astype("Int64")

        if is_lon:
            d = (v // 1_000_000).astype(float)          # DDD
            m = ((v // 10_000) % 100).astype(float)     # MM
            sec = ((v % 10_000) / 100.0).astype(float)  # SS.ss
            dd = d + m/60.0 + sec/3600.0
            out.loc[mask_packed] = -dd                  # West negative
        else:
            d = (v // 1_000_000).astype(float)          # DD
            m = ((v // 10_000) % 100).astype(float)     # MM
            sec = ((v % 10_000) / 100.0).astype(float)  # SS.ss
            out.loc[mask_packed] = d + m/60.0 + sec/3600.0

    return out


def safe_year(series: pd.Series) -> pd.Series:
    return pd.to_numeric(series, errors="coerce").astype("Int64")

def make_bridge_uid(location_series: pd.Series, lat: pd.Series, lon: pd.Series) -> pd.Series:
    loc = location_series.fillna("").astype(str).str.strip()
    lat_s = lat.round(6).astype(str)
    lon_s = lon.round(6).astype(str)
    return loc + "_" + lat_s + "_" + lon_s

# -----------------------------
# 2) Load bridges (TX25) as strings (prevents scientific notation)
# -----------------------------
tx = pd.read_csv(BRIDGE_CSV, low_memory=False, dtype=str)

loc_col  = pick_col(tx, [r"LOCATION_009", r"\bLOCATION\b"], required=True)
lat_col  = pick_col(tx, [r"\bLAT(_016)?\b", r"LATITUDE"], required=True)
lon_col  = pick_col(tx, [r"\bLONG(_017)?\b", r"\bLON(GITUDE)?\b"], required=True)
year_col = pick_col(tx, [r"YEAR_BUILT_027", r"YEAR[_\s]?BUILT", r"YR[_\s]?BUILT"], required=False)

tx["_lat"] = to_decimal_degrees(tx[lat_col], is_lon=False)
tx["_lon"] = to_decimal_degrees(tx[lon_col], is_lon=True)
tx["_year_built"] = safe_year(tx[year_col]) if year_col is not None else pd.Series([pd.NA]*len(tx), dtype="Int64")

tx = tx[pd.notna(tx["_lat"]) & pd.notna(tx["_lon"])].copy()
tx = tx[tx["_lat"].between(25.0, 37.5) & tx["_lon"].between(-107.5, -92.5)].copy()  # TX-ish bounds

tx["bridge_uid"] = make_bridge_uid(tx[loc_col], tx["_lat"], tx["_lon"])

# enforce uniqueness if any duplicates exist
if tx["bridge_uid"].duplicated().any():
    tx["bridge_uid"] = tx["bridge_uid"] + "__" + tx.groupby("bridge_uid").cumcount().astype(str)

print(f"Bridges used: {len(tx):,} | unique bridge_uid: {tx['bridge_uid'].nunique():,}")

bridge_rad = np.deg2rad(tx[["_lat", "_lon"]].to_numpy())

# -----------------------------
# 3) Load tornado points and filter 1950–2010 + Texas
# -----------------------------
tor = pd.read_csv(TORNADO_CSV, low_memory=False)

t_lat_col  = pick_col(tor, [r"\bLAT\b", r"LATITUDE", r"BEGIN_LAT", r"START_LAT", r"SLAT", r"TOUCHDOWN_LAT"])
t_lon_col  = pick_col(tor, [r"\bLON\b", r"\bLONG\b", r"LONGITUDE", r"BEGIN_LON", r"START_LON", r"SLON", r"TOUCHDOWN_LON"])
t_year_col = pick_col(tor, [r"\bYEAR\b", r"BEGIN_YEAR", r"EVENT_YEAR"], required=False)
t_date_col = pick_col(tor, [r"DATE", r"BEGIN_DATE", r"EVENT_DATE", r"START_DATE"], required=False)

tor["_tlat"] = to_decimal_degrees(tor[t_lat_col], is_lon=False)
tor["_tlon"] = to_decimal_degrees(tor[t_lon_col], is_lon=True)

if t_year_col is not None:
    tor["_year"] = safe_year(tor[t_year_col])
elif t_date_col is not None:
    dt = pd.to_datetime(tor[t_date_col], errors="coerce")
    tor["_year"] = dt.dt.year.astype("Int64")
else:
    raise ValueError("Tornado file must have YEAR or DATE column.")

tor = tor[(tor["_year"] >= YEAR_START) & (tor["_year"] <= YEAR_END)].copy()
tor = tor[pd.notna(tor["_tlat"]) & pd.notna(tor["_tlon"])].copy()

# Texas filter by state if available; else bbox
t_state_col = pick_col(tor, [r"\bSTATE\b", r"STATE_ABBR", r"\bST\b"], required=False)
if t_state_col is not None:
    st = tor[t_state_col].astype(str).str.upper()
    tor = tor[(st.eq("TX")) | (st.eq("TEXAS"))].copy()
else:
    tor = tor[tor["_tlat"].between(25.0, 37.5) & tor["_tlon"].between(-107.5, -92.5)].copy()

# de-duplicate to avoid inflated N_i
event_id_col = pick_col(tor, [r"EVENT_ID", r"EPISODE_ID", r"TORNADO_ID"], required=False)
if event_id_col is not None:
    tor = tor.drop_duplicates(subset=[event_id_col])
else:
    tor = tor.drop_duplicates(subset=["_year", "_tlat", "_tlon"])

print(f"Tornado points used (TX) {YEAR_START}-{YEAR_END}: {len(tor):,}")

tornado_rad = np.deg2rad(tor[["_tlat", "_tlon"]].to_numpy())

# -----------------------------
# 4) Count tornado points within 1 mile using BallTree (FAST)
# -----------------------------
tree = BallTree(tornado_rad, metric="haversine")
idx_list = tree.query_radius(bridge_rad, r=RADIUS_RAD, return_distance=False)
tx["N_i"] = np.fromiter((len(ix) for ix in idx_list), dtype=int, count=len(idx_list))

# -----------------------------
# 5) Compute p_tornado, V_age, Risk
# -----------------------------
tx["lambda_i"]  = tx["N_i"] / float(T_YEARS)
tx["p_tornado"] = 1.0 - np.exp(-tx["lambda_i"])

tx["V_age"] = 1.0
mask_y = pd.notna(tx["_year_built"])
tx.loc[mask_y, "V_age"] = 1.0 + (1990.0 - tx.loc[mask_y, "_year_built"].astype(float)) / 35.0
if CLIP_VAGE_MIN_1:
    tx["V_age"] = tx["V_age"].clip(lower=1.0)

tx["Risk_tornado"] = tx["p_tornado"] * tx["V_age"]
tx["Risk_rank_desc"] = tx["Risk_tornado"].rank(ascending=False, method="min").astype(int)
tx["coordinates (lat, lon)"] = tx["_lat"].round(6).astype(str) + ", " + tx["_lon"].round(6).astype(str)

out = tx[[
    "bridge_uid", loc_col, "coordinates (lat, lon)", "_year_built",
    "N_i", "lambda_i", "p_tornado", "V_age", "Risk_tornado", "Risk_rank_desc"
]].sort_values("Risk_tornado", ascending=False)

out.to_csv(OUT_CSV, index=False)

print(f"\nSaved: {OUT_CSV}")
print(out.head(10))


Bridges used: 56,951 | unique bridge_uid: 56,951
Tornado points used (TX) 1950-2010: 7,556

Saved: tornadoo_risk_1950_2010.csv
                                            bridge_uid  \
25522     '1.10 MI S  OF LP 336'_30.320236_-95.4644__0   
28945    '0.25 MI S  OF SH 60'_29.308164_-96.103922__0   
9021     '0.40 MI S OF US 70'_34.178933_-101.702919__0   
21503   '0.10 MI W  OF SH 288'_29.732839_-95.375328__0   
25615    '2.00 MI E  OF IH 45'_30.316147_-95.440314__0   
26210  '0.36 MI N OF LEWIS ST'_30.321183_-95.460314__0   
183        '0.3 MI S OF SH 56'_33.572306_-96.178161__0   
47999      '0.4 MI N OF SH 31'_32.098378_-96.474403__0   
38477      '1.9 MI S OF IH 37'_27.771294_-97.415406__0   
38476      '1.9 MI S OF IH 37'_27.771214_-97.415242__0   

                  LOCATION_009  coordinates (lat, lon)  _year_built  N_i  \
25522   '1.10 MI S  OF LP 336'     30.320236, -95.4644         1926    8   
28945    '0.25 MI S  OF SH 60'   29.308164, -96.103922         1930    8   
9021  

### **Over-height risk**

In [ ]:
import pandas as pd
import numpy as np

# =========================
# 0) INPUT / OUTPUT
# =========================
INPUT_FILE = "tx25-final.csv"
OUT_ALL    = "bridges_overheight_risk_all.csv"
OUT_TOP25  = "top_25_overheight_risk.csv"

# =========================
# 1) NBI coordinate helpers
# =========================
def nbi_lat_to_dd(x):
    """LAT_016: DDMMSSss -> decimal degrees."""
    if pd.isna(x):
        return np.nan
    x = int(float(x))
    d = x // 1_000_000
    m = (x // 10_000) % 100
    s = (x % 10_000) / 100.0
    return d + m / 60.0 + s / 3600.0

def nbi_lon_to_dd(x):
    """LONG_017: DDDMMSSss -> decimal degrees, West negative."""
    if pd.isna(x):
        return np.nan
    x = int(float(x))
    d = x // 1_000_000
    m = (x // 10_000) % 100
    s = (x % 10_000) / 100.0
    return -(d + m / 60.0 + s / 3600.0)

# =========================
# 2) FDOT Step-3 detour models
# =========================
def d_vc_interstate(vc_ft):
    """FDOT over-height detour % for Interstate functional classes."""
    if pd.isna(vc_ft):
        return np.nan
    vc = float(vc_ft)
    if vc < 9.65:
        return 100.0
    elif 9.65 <= vc < 13.0:
        return 855.91 - 223.43 * vc + 22.199 * (vc**2) - 0.74236 * (vc**3)
    elif 13.0 <= vc < 14.0:
        return 1.0956e56 * (vc ** (-48.683))
    elif 14.0 <= vc < 16.1:
        return 14.567 - 0.9046 * vc
    else:
        return 0.0

def d_vc_other(vc_ft):
    """FDOT over-height detour % for non-Interstate functional classes."""
    if pd.isna(vc_ft):
        return np.nan
    vc = float(vc_ft)
    if vc < 7.3:
        return 100.0
    elif 7.3 <= vc < 13.5:
        return -26.275 + 34.692 * vc - 2.3894 * (vc**2)
    elif 13.5 <= vc < 14.0:
        return 138.86 - 9.886 * vc
    else:
        return 0.0

# =========================
# 3) Pick a VC column robustly
# =========================
def pick_vc_column(df):
    # Edit priority if you want a specific one only
    candidates = ["VERT_CLR_OVER_010", "VERT_CLR_UND_054B", "MIN_VERT_CLR_010"]
    for c in candidates:
        if c in df.columns:
            return c
    return None

def normalize_vc_to_ft(series):
    """
    Convert VC to feet if needed.
    Heuristic: if median < 8 -> likely meters -> convert to ft; else assume feet.
    """
    s = pd.to_numeric(series, errors="coerce")
    med = s.dropna().median() if s.notna().any() else np.nan
    if pd.isna(med):
        return s
    if med < 8:
        return s / 0.3048
    return s

# =========================
# 4) LOAD DATA
# =========================
df = pd.read_csv(INPUT_FILE, low_memory=False)

# =========================
# 5) STEP 2 inputs: VC, ADT, truck fraction, functional class
# =========================
vc_col = pick_vc_column(df)
if vc_col is None:
    raise ValueError(
        "No vertical clearance column found. Expected one of: "
        "VERT_CLR_OVER_010, VERT_CLR_UND_054B, MIN_VERT_CLR_010"
    )

df["VC_ft"] = normalize_vc_to_ft(df[vc_col])

df["ADT"] = pd.to_numeric(df.get("ADT_029", np.nan), errors="coerce").fillna(0)
df["truck_frac"] = (
    pd.to_numeric(df.get("PERCENT_ADT_TRUCK_109", np.nan), errors="coerce")
    .fillna(0) / 100.0
).clip(0, 1)

fc = pd.to_numeric(df.get("FUNCTIONAL_CLASS_026", np.nan), errors="coerce")
df["is_interstate"] = fc.isin([1, 2, 11, 12])  # keep if matches your TX coding

# =========================
# 6) STEP 3: detour percentage d(VC)
# =========================
df["d_vc"] = np.where(
    df["is_interstate"],
    df["VC_ft"].apply(d_vc_interstate),
    df["VC_ft"].apply(d_vc_other)
)
df["d_vc"] = pd.to_numeric(df["d_vc"], errors="coerce").clip(0, 100)

# =========================
# 7) STEP 4: exposure proxy N_detour
# =========================
df["N_detour"] = (df["d_vc"] / 100.0) * df["ADT"] * df["truck_frac"]

# =========================
# 8) STEP 5: likelihood proxy p_height ∝ N_detour
# =========================
df["p_height"] = df["N_detour"]

# =========================
# 9) STEP 6: vulnerability multiplier (your reminder)
# =========================
df["V_elem_height"] = 1.0

# =========================
# 10) STEP 7: Risk Index
# =========================
df["Risk_Index"] = df["p_height"] * df["V_elem_height"]

# =========================
# 11) Create combined coordinates (lat, lon)
#     - Convert NBI LAT_016/LONG_017 to decimal degrees
#     - Combine into a single string column:
#       "32.027383, -97.095483"
# =========================
if "LAT_016" in df.columns and "LONG_017" in df.columns:
    df["LAT_DD"] = pd.to_numeric(df["LAT_016"], errors="coerce").apply(nbi_lat_to_dd)
    df["LON_DD"] = pd.to_numeric(df["LONG_017"], errors="coerce").apply(nbi_lon_to_dd)

    def to_coord_str(row):
        lat, lon = row["LAT_DD"], row["LON_DD"]
        if pd.isna(lat) or pd.isna(lon):
            return ""
        return f"{lat:.6f}, {lon:.6f}"

    df["coordinates (lat, lon)"] = df.apply(to_coord_str, axis=1)

# =========================
# 12) Rank + Save ALL columns (as you asked) + Top 25
# =========================
df_ranked = df.sort_values("Risk_Index", ascending=False).copy()

# Save ALL bridges with ALL columns kept (including new computed ones)
df_ranked.to_csv(OUT_ALL, index=False)

# Save TOP 25 (also with ALL columns)
top_25 = (
    df_ranked[df_ranked["ADT"] > 0]
    .head(25)
    .copy()
)
top_25.to_csv(OUT_TOP25, index=False)

print(f"VC column used: {vc_col}")
print(f"Saved ALL ranked bridges (all columns): {OUT_ALL}")
print(f"Saved TOP 25 bridges (all columns):     {OUT_TOP25}")

# Preview coordinates column if created
if "coordinates (lat, lon)" in df_ranked.columns:
    print("\nPreview of coordinates (lat, lon):")
    print(top_25[["STRUCTURE_NUMBER_008", "coordinates (lat, lon)", "Risk_Index"]].head(10).to_string(index=False))


VC column used: VERT_CLR_UND_054B
Saved ALL ranked bridges (all columns): bridges_overheight_risk_all.csv
Saved TOP 25 bridges (all columns):     top_25_overheight_risk.csv

Preview of coordinates (lat, lon):
STRUCTURE_NUMBER_008 coordinates (lat, lon)  Risk_Index
   91100001424489.00  32.027383, -97.095483   291639.60
  142460001508247.00  30.625933, -97.691122    72415.84
   91610001408506.00  31.724464, -97.102842    62480.16
   22200008112086.00  32.899894, -97.317233    61265.82
   90140001506537.00  31.011375, -97.485569    40989.48
   91100001424205.00  32.014528, -97.096150    33886.40
  121020027107626.00  29.784178, -95.494297    31830.84
  100370045001028.00  31.894792, -95.036664    31356.99
  121700067508229.00  30.293508, -95.464000    31160.00
  121020027107610.00  29.784439, -95.553464    30979.04


### **overloads risk**

In [ ]:
import pandas as pd
import numpy as np

# ----------------------------
# FDOT Overload Risk Screening
# ----------------------------


# ---------- Helper functions ----------
def detour_percent_overload(ri_lb, is_interstate):
    """FDOT overload detour percent d_i (Eq. 9.1 & 9.2)."""
    if pd.isna(ri_lb):
        return np.nan

    if is_interstate:
        # FDOT Eq. (9.1) – Interstate
        if ri_lb < 10000:
            return 100.0
        elif ri_lb < 80000:
            return 102.24 - (8.982e-5 * ri_lb) - (1.4336e-8 * ri_lb**2)
        elif ri_lb < 91000:
            return 18.976 - (2.083e-4 * ri_lb)
        else:
            return 0.0
    else:
        # FDOT Eq. (9.2) – Non-Interstate
        if ri_lb < 3725:
            return 100.0
        elif ri_lb < 85000:
            return (
                107.26
                - (1.9743e-3 * ri_lb)
                + (6.5265e-9 * ri_lb**2)
                + (2.2256e-14 * ri_lb**3)
            )
        else:
            return 0.0

def v_age_overload(year_built):
    """FDOT age multiplier (Eq. 9.5)."""
    if pd.isna(year_built):
        return 1.0
    if year_built >= 1998:
        return 0.75
    elif 1970 < year_built < 1998:
        return 1.0 + (1990 - year_built) / 35.0
    else:
        return 1.25

def v_elem_overload(super_cond):
    """FDOT element multiplier using superstructure condition (Eq. 9.6)."""
    if pd.isna(super_cond):
        return 1.0
    if super_cond >= 9:
        return 0.85
    elif 1 < super_cond < 9:
        return 1.0 + (7 - super_cond) / 40.0
    else:
        return 1.15

# ---------- Load data ----------
df = pd.read_csv(INPUT_FILE, low_memory=False)

# ---------- STEP 1: Inputs ----------
df["ADT"] = pd.to_numeric(df["ADT_029"], errors="coerce").fillna(0)

df["truck_frac"] = (
    pd.to_numeric(df["PERCENT_ADT_TRUCK_109"], errors="coerce")
    .fillna(0) / 100.0
).clip(0, 1)

df["year_built"] = pd.to_numeric(df["YEAR_BUILT_027"], errors="coerce")
df["super_cond"] = pd.to_numeric(df["SUPERSTRUCTURE_COND_059"], errors="coerce")

# Inventory / operating rating in tons → convert to pounds
df["r_i_lb"] = pd.to_numeric(df["INVENTORY_RATING_066"], errors="coerce") * 2000.0

# Interstate identification (TX functional class coding)
fc = pd.to_numeric(df["FUNCTIONAL_CLASS_026"], errors="coerce")
df["is_interstate"] = fc.isin([1, 2, 11, 12])

# ---------- STEP 2: Detour percentage ----------
df["d_i"] = [
    detour_percent_overload(r, is_int)
    for r, is_int in zip(df["r_i_lb"], df["is_interstate"])
]
df["d_i"] = df["d_i"].clip(0, 100)

# ---------- STEP 3–4: Exposure & likelihood ----------
df["N_i"] = (df["d_i"] / 100.0) * df["ADT"] * df["truck_frac"]
df["p_overload"] = df["N_i"]   # proportional likelihood

# ---------- STEP 5: Vulnerability multipliers ----------
df["V_age_overload"] = df["year_built"].apply(v_age_overload)
df["V_elem_overload"] = df["super_cond"].apply(v_elem_overload)

# ---------- STEP 6: Final overload risk index ----------
df["Risk_Index"] = (
    df["p_overload"]
    * df["V_age_overload"]
    * df["V_elem_overload"]
)

# ---------- Rank bridges ----------
df_ranked = df.sort_values("Risk_Index", ascending=False)

# ---------- Save outputs ----------
# 1) ALL bridges ranked by overload risk
df_ranked.to_csv("bridges_overload_risk_all.csv", index=False)

# 2) TOP 25 bridges by overload risk
top_25 = (
    df_ranked[df_ranked["ADT"] > 0]
    .head(25)
    .copy()
)


top_25.to_csv("top_25_overload_risk.csv", index=False)


# ---------- Display ----------
print("Top 25 Bridges by FDOT Overload Risk Index:\n")
print(
    top_25[
        [
            "STRUCTURE_NUMBER_008",
            "LOCATION_009",
            "Risk_Index",
            "ADT",
            "truck_frac",
            "r_i_lb",
            "d_i"
        ]
    ].to_string(index=False)
)

Top 25 Bridges by FDOT Overload Risk Index:

STRUCTURE_NUMBER_008                LOCATION_009   Risk_Index    ADT  truck_frac  r_i_lb       d_i
   91100001424489.00         '1.2 MI S OF US 77' 76661.273824 810110        0.36 65400.0 35.048406
   22200008112086.00 '.9 MI N OF US287 OVER I35' 50591.921706 185654        0.33 47200.0 66.062182
   61650000514169.00           'IH 20 at SH 349' 38474.321139 244794        0.35 65400.0 35.048406
  142460001508247.00         '0.3 MI S OF SH 29' 32518.890912 517256        0.14 65400.0 35.048406
  180570000911347.00    '0.3 Mi E of Preston St' 32049.776120 194353        0.17 40000.0 75.709600
  180570237403145.00       '0.7 MI  W OF SH 342' 31027.859030 152210        0.20 47200.0 66.062182
  180570237403190.00        '0.85 MI E OF IH 45' 29739.349772 138942        0.21 47200.0 66.062182
  180570237403191.00         '1.6 MI E OF IH 45' 29739.349772 138942        0.21 47200.0 66.062182
   91100001424205.00 '2.5 MI S OF IH 35E/IH 35W' 27982.618950  8

### **flood scour risk**

In [ ]:
import pandas as pd
import numpy as np

# =========================
# 0) INPUT / OUTPUT
# =========================
INPUT_FILE = "TX-25.csv"
OUT_ALL    = "flood_scour_risk_all.csv"
OUT_TOP25  = "top_25_flood_scour_risk.csv"

# =========================
# 1) NBI coordinate helpers
# =========================
def nbi_lat_to_dd(x):
    """LAT_016: DDMMSSss -> decimal degrees."""
    if pd.isna(x):
        return np.nan
    x = int(float(x))
    d = x // 1_000_000
    m = (x // 10_000) % 100
    s = (x % 10_000) / 100.0
    return d + m / 60.0 + s / 3600.0

def nbi_lon_to_dd(x):
    """LONG_017: DDDMMSSss -> decimal degrees, West negative."""
    if pd.isna(x):
        return np.nan
    x = int(float(x))
    d = x // 1_000_000
    m = (x // 10_000) % 100
    s = (x % 10_000) / 100.0
    return -(d + m / 60.0 + s / 3600.0)

# =========================
# 2) FDOT probability mapping for pflood
#    (FDOT uses discrete probability levels 0.01, 0.02, 0.10, 0.50)
#    from Waterway Adequacy (Item 71) overtopping frequency categories.
# =========================
def pflood_from_item71(item71):
    """
    Convert NBI Item 71 (Waterway Adequacy) to annual overtopping probability.
    This matches the memo's discrete mapping (Remote/Slight/Occasional/Frequent)
    -> {0.01, 0.02, 0.10, 0.50}.
    """
    if pd.isna(item71):
        return np.nan

    # Handle textual "N" (not over waterway)
    s = str(item71).strip().upper()
    if s == "N":
        return 0.0  # not over waterway => no overtopping probability used

    # Numeric codes
    try:
        v = int(float(s))
    except Exception:
        return np.nan

    # Bridge closed or invalid
    if v == 0:
        # FDOT table treats closed separately; set to 0 to avoid ranking closed bridges high
        return 0.0

    # FDOT discrete levels (engineering screening)
    # 9 -> Remote (0.01)
    # 6-8 -> Slight (0.02)
    # 3-5 -> Occasional (0.10)
    # 1-2 -> Frequent (0.50)
    if v == 9:
        return 0.01
    elif v in (6, 7, 8):
        return 0.02
    elif v in (3, 4, 5):
        return 0.10
    elif v in (1, 2):
        return 0.50
    else:
        return np.nan

# =========================
# 3) Vulnerability multipliers
# =========================
def v_chan_from_item61(item61):
    """Eq. (5.7): channel vulnerability multiplier from NBI Item 61."""
    if pd.isna(item61):
        return 1.000
    s = str(item61).strip().upper()
    if s == "N":
        return 1.000
    try:
        v = int(float(s))
    except Exception:
        return 1.000

    if v in (8, 9):
        return 0.400
    elif v == 7:
        return 0.995
    elif v == 6:
        return 1.139
    elif v in (4, 5):
        return 1.600
    elif v in (0, 1, 2, 3):
        return 2.995
    else:
        return 1.000

def v_scour_from_item113(item113):
    """Eq. (5.8): scour criticality multiplier from NBI Item 113."""
    if pd.isna(item113):
        return 0.333  # treat missing similar to N (conservative neutral-low)
    s = str(item113).strip().upper()

    if s == "N":
        return 0.333
    if s == "U":
        return 2.000

    # numeric codes 0-9
    try:
        v = int(float(s))
    except Exception:
        return 0.333

    if v in (0, 1, 2, 3):
        return 2.000
    elif v in (4, 5, 6, 7, 8, 9):
        return 1.200
    else:
        return 0.333

def v_age_from_yearbuilt(year_built):
    """Eq. (5.9): V_age = 1 + (1990 - Y)/35."""
    if pd.isna(year_built):
        return 1.0
    try:
        y = float(year_built)
    except Exception:
        return 1.0
    return 1.0 + (1990.0 - y) / 35.0

# =========================
# 4) Load data
# =========================
df = pd.read_csv(INPUT_FILE, low_memory=False)

# =========================
# 5) Required NBI items (robust column picking)
#    You may adjust these if your TX25 uses slightly different names.
# =========================
# NBI Item 71: Waterway Adequacy
item71_col = "WATERWAY_ADEQUACY_071" if "WATERWAY_ADEQUACY_071" in df.columns else "WATERWAY_ADEQUACY_071A" if "WATERWAY_ADEQUACY_071A" in df.columns else "WATERWAY_ADEQUACY_071"  # keep fallback
# If your TX25 uses the common naming "WATERWAY_ADEQUACY_071", this works.
# If not found, we will attempt a softer fallback below.

# Soft fallback for Item 71
if item71_col not in df.columns:
    candidates71 = [c for c in df.columns if "WATERWAY" in c.upper() and "71" in c]
    if len(candidates71) == 0:
        raise ValueError("Could not find NBI Item 71 (Waterway Adequacy) column in TX25.csv.")
    item71_col = candidates71[0]

# NBI Item 26: Functional Class (used in memo table; we keep it but the simplified pflood mapping uses Item 71 levels)
fc_col = "FUNCTIONAL_CLASS_026" if "FUNCTIONAL_CLASS_026" in df.columns else None
if fc_col is None:
    raise ValueError("Could not find FUNCTIONAL_CLASS_026 in TX25.csv.")

# NBI Item 61: Channel Condition
ch61_col = "CHANNEL_COND_061" if "CHANNEL_COND_061" in df.columns else None
if ch61_col is None:
    candidates61 = [c for c in df.columns if "CHANNEL" in c.upper() and "61" in c]
    if len(candidates61) == 0:
        raise ValueError("Could not find NBI Item 61 (Channel Condition) column in TX25.csv.")
    ch61_col = candidates61[0]

# NBI Item 113: Scour Criticality
scour113_col = "SCOUR_CRIT_113" if "SCOUR_CRIT_113" in df.columns else None
if scour113_col is None:
    candidates113 = [c for c in df.columns if ("SCOUR" in c.upper() and "113" in c) or ("CRIT" in c.upper() and "113" in c)]
    if len(candidates113) == 0:
        raise ValueError("Could not find NBI Item 113 (Scour Criticality) column in TX25.csv.")
    scour113_col = candidates113[0]

# NBI Item 27: Year Built
yb_col = "YEAR_BUILT_027" if "YEAR_BUILT_027" in df.columns else None
if yb_col is None:
    raise ValueError("Could not find YEAR_BUILT_027 in TX25.csv.")

# =========================
# 6) Step 2: I_overwater from Item 71 (Eq. 5.1)
# =========================
def i_overwater_from_item71(v):
    if pd.isna(v):
        return 1  # if missing, don't auto-zero; keep bridge in screening
    s = str(v).strip().upper()
    return 0 if s == "N" else 1

df["I_overwater"] = df[item71_col].apply(i_overwater_from_item71)

# =========================
# 7) Step 3: pflood from Item 71 (+ table logic)
# =========================
df["p_flood"] = df[item71_col].apply(pflood_from_item71).fillna(0.0)

# =========================
# 8) Step 4/7/8/9: vulnerability multipliers
# =========================
# Step 5 (Element vulnerability): NBI-only -> set to 1 (your requested approach)
df["V_elem_flood"] = 1.0

# Step 6 (Design vulnerability): Table 5.5 requires an agency lookup; set to 1 for NBI-only screening
df["V_des_flood"] = 1.0

# Step 7 (Channel vulnerability): Eq. (5.7)
df["V_chan_flood"] = df[ch61_col].apply(v_chan_from_item61)

# Step 8 (Scour criticality vulnerability): Eq. (5.8)
df["V_scour_flood"] = df[scour113_col].apply(v_scour_from_item113)

# Step 9 (Age vulnerability): Eq. (5.9)
df["V_age_flood"] = pd.to_numeric(df[yb_col], errors="coerce").apply(v_age_from_yearbuilt)

# =========================
# 9) Step 10: final Risk index (Eq. 5.11)
# =========================
df["Risk_flood"] = (
    df["I_overwater"]
    * df["p_flood"]
    * df["V_scour_flood"]
    * df["V_elem_flood"]
    * df["V_age_flood"]
    * df["V_des_flood"]
    * df["V_chan_flood"]
)

# =========================
# 10) Add coordinates (lat, lon) column like you want
# =========================
if "LAT_016" in df.columns and "LONG_017" in df.columns:
    df["LAT_DD"] = pd.to_numeric(df["LAT_016"], errors="coerce").apply(nbi_lat_to_dd)
    df["LON_DD"] = pd.to_numeric(df["LONG_017"], errors="coerce").apply(nbi_lon_to_dd)

    def to_coord_str(row):
        lat, lon = row["LAT_DD"], row["LON_DD"]
        if pd.isna(lat) or pd.isna(lon):
            return ""
        return f"{lat:.6f}, {lon:.6f}"

    df["coordinates (lat, lon)"] = df.apply(to_coord_str, axis=1)

# =========================
# 11) Rank + save ALL columns + top 25
# =========================
df_ranked = df.sort_values("Risk_flood", ascending=False).copy()

df_ranked.to_csv(OUT_ALL, index=False)

top_25 = df_ranked.head(25).copy()
top_25.to_csv(OUT_TOP25, index=False)

print("Done ✅")
print(f"Saved ALL bridges ranked: {OUT_ALL}")
print(f"Saved TOP 25 bridges:    {OUT_TOP25}")
print("\nTop 10 preview (ID, Risk_flood, coordinates if present):\n")

cols_preview = ["STRUCTURE_NUMBER_008", "Risk_flood"]
if "coordinates (lat, lon)" in df_ranked.columns:
    cols_preview.append("coordinates (lat, lon)")
print(top_25[cols_preview].head(10).to_string(index=False))


Done ✅
Saved ALL bridges ranked: flood_scour_risk_all.csv
Saved TOP 25 bridges:    top_25_flood_scour_risk.csv

Top 10 preview (ID, Risk_flood, coordinates if present):

STRUCTURE_NUMBER_008  Risk_flood coordinates (lat, lon)
     222330B00005001    4.342857 29.354033, -100.895750
     080770AA0239001    2.331429 32.836383, -100.211569
     080770AA0212001    2.331429 32.647067, -100.520386
     081280AA0265002    2.057143  32.778103, -99.689228
     081680000601151    1.837714 32.409517, -100.750925
     180570AA0222001    1.554286  32.563878, -96.624303
     150830084904004    1.449857  29.078156, -98.864292
     151330082904001    1.444903  30.067006, -99.383919
     222540087805006    1.425377  28.692008, -99.808106
     100010012204025    1.405851  31.940558, -95.944258


final tornado

In [ ]:
import pandas as pd
import numpy as np
import re
from sklearn.neighbors import BallTree

# ============================================================
# 1) CONFIGURATION & PATHS
# ============================================================
BRIDGE_CSV  = "TX25.csv"
TORNADO_CSV = "1950-2024_all_tornadoes.csv"
OUT_CSV     = "tornado_risk_assessment_fdot.csv"

# FDOT Analysis window: 1950-2010 (61 years inclusive)
YEAR_START = 1950
YEAR_END   = 2010
T_YEARS    = (YEAR_END - YEAR_START + 1)

BUFFER_MILES = 1.0
EARTH_RADIUS_M = 6371008.8
RADIUS_M = BUFFER_MILES * 1609.344
RADIUS_RAD = RADIUS_M / EARTH_RADIUS_M

# ============================================================
# 2) COORDINATE & DESIGN HELPERS
# ============================================================
def to_decimal_degrees(series: pd.Series, is_lon=False) -> pd.Series:
    """Converts NBI packed format (DDMMSSss) or raw DD to Decimal Degrees."""
    s = pd.to_numeric(series, errors="coerce")
    out = pd.Series(np.nan, index=series.index, dtype="float64")

    # Already Decimal Degrees
    mask_dd = s.abs() < 180
    out.loc[mask_dd] = s.loc[mask_dd]
    if is_lon:
        out.loc[mask_dd & (out > 0)] = -out.loc[mask_dd & (out > 0)]

    # NBI Packed Format (DDMMSSss)
    mask_packed = out.isna() & s.notna()
    if mask_packed.any():
        v = s.loc[mask_packed].abs().astype("Int64")
        d = (v // 1_000_000).astype(float)
        m = ((v // 10_000) % 100).astype(float)
        sec = ((v % 10_000) / 100.0).astype(float)
        dd = d + m/60.0 + sec/3600.0
        out.loc[mask_packed] = -dd if is_lon else dd
    return out

def get_v_des_score(type_code, underclearance):
    """Maps NBI Item 43B and 54B to Table 12.3 Scores (s_des)."""
    try: tc = int(float(type_code))
    except: return 1 # Default

    is_high = False
    try:
        # Check if vertical underclearance is > 6.1m (~20ft)
        if float(underclearance) > 6.1: is_high = True
    except: pass

    # Table 12.3: {NBI_43B_Code: (Low_Score, High_Score)}
    score_map = {
        1: (1, 2), 2: (1, 2), 3: (1, 2), 4: (1, 2), 5: (1, 2), 6: (1, 2),
        7: (1, 2), 8: (1, 2), 9: (4, 5), 10:(4, 5), 11:(1, 2), 12:(1, 2),
        13:(4, 4), 14:(4, 4), 15:(3, 4), 16:(3, 4), 17:(3, 4), 18:(2, 4),
        19:(2, 3), 20:(2, 2), 21:(1, 2), 22:(1, 2)
    }
    scores = score_map.get(tc, (1, 1))
    return scores[1] if is_high else scores[0]

# ============================================================
# 3) PROCESSING
# ============================================================
print("Loading datasets...")
tx = pd.read_csv(BRIDGE_CSV, low_memory=False, dtype=str)
tor = pd.read_csv(TORNADO_CSV, low_memory=False)

# Clean Bridge Coordinates
tx["_lat"] = to_decimal_degrees(tx["LAT_016"], is_lon=False)
tx["_lon"] = to_decimal_degrees(tx["LONG_017"], is_lon=True)
tx = tx.dropna(subset=["_lat", "_lon"]).copy()

# Filter Tornadoes (Texas, 1950-2010)
tor_tx = tor[(tor["st"] == "TX") & (tor["yr"] >= YEAR_START) & (tor["yr"] <= YEAR_END)].copy()

# --- Step 1 & 2: Likelihood (p_tornado) ---
print("Calculating spatial likelihood...")
bridge_rad = np.deg2rad(tx[["_lat", "_lon"]].to_numpy())
tornado_rad = np.deg2rad(tor_tx[["slat", "slon"]].to_numpy())

tree = BallTree(tornado_rad, metric="haversine")
idx_list = tree.query_radius(bridge_rad, r=RADIUS_RAD, return_distance=False)

tx["N_i"] = [len(ix) for ix in idx_list]
tx["lambda_i"] = tx["N_i"] / float(T_YEARS)
tx["p_tornado"] = 1.0 - np.exp(-tx["lambda_i"])

# --- Step 3: Age Vulnerability (V_age) ---
tx["Y"] = pd.to_numeric(tx["YEAR_BUILT_027"], errors="coerce")
tx["V_age"] = (1.0 + (1990.0 - tx["Y"]) / 35.0).clip(lower=1.0).fillna(1.0)

# --- Step 4: Design Vulnerability (V_des) ---
tx["s_des"] = tx.apply(lambda r: get_v_des_score(r["STRUCTURE_TYPE_043B"], r["VERT_CLR_UND_054B"]), axis=1)
tx["V_des"] = (tx["s_des"] + 1.0) / 3.0

# --- Step 5: Element Vulnerability (V_elem) ---
# Default to 1.0 (Moderate) as element-level tables are separate from NBI CSV
tx["V_elem"] = 1.0

# --- Final Calculation: Risk ---
tx["Risk_tornado"] = tx["p_tornado"] * tx["V_age"] * tx["V_des"] * tx["V_elem"]
tx["Risk_Rank"] = tx["Risk_tornado"].rank(ascending=False, method="min").astype(int)

# ============================================================
# 4) SAVE RESULTS
# ============================================================
output_cols = [
    "STRUCTURE_NUMBER_008", "LOCATION_009", "_lat", "_lon", "YEAR_BUILT_027",
    "N_i", "p_tornado", "V_age", "V_des", "Risk_tornado", "Risk_Rank"
]
result = tx[output_cols].sort_values("Risk_tornado", ascending=False)
result.to_csv(OUT_CSV, index=False)

print(f"Success! Assessment saved to {OUT_CSV}")
print(result.head(10))

Loading datasets...
Calculating spatial likelihood...
Success! Assessment saved to tornado_risk_assessment_fdot.csv
      STRUCTURE_NUMBER_008           LOCATION_009       _lat       _lon  \
36451      150950053502121   '1.10 MI E OF SH 80'  29.650008 -97.641575   
36439      150950053502109    '4.0 MI E OF US 90'  29.612517 -97.859561   
36440      150950053502110    '5.0 MI E OF US 90'  29.614867 -97.844992   
36441      150950053502111   '5.00 MI E OF US 90'  29.614603 -97.844992   
36442      150950053502112   '5.00 MI E OF US 90'  29.614311 -97.844997   
36443      150950053502113    '5.9 MI E OF US 90'  29.614975 -97.832083   
36444      150950053502114    '5.9 MI E OF US 90'  29.614706 -97.832339   
36445      150950053502115    '6.0 MI E OF US 90'  29.615442 -97.830211   
36446      150950053502116    '6.1 MI E OF US 90'  29.614897 -97.825228   
36447      150950053502117  '0.5 MI E OF FM 2438'  29.615044 -97.811061   

      YEAR_BUILT_027  N_i  p_tornado  V_age     V_des  Ris

# Updated Hurricane risk

In [ ]:
import numpy as np
import pandas as pd
from sklearn.neighbors import BallTree

# ============================================================
# INPUT FILES (your two datasets)
# ============================================================
BRIDGE_FILE  = "/mnt/data/_MConverter.eu_TX25 (1).csv"
IBTRACS_FILE = "/mnt/data/IBTrACS_TEXAS_ONLY.csv"
OUTPUT_FILE  = "texas_hurricane_risk_ranked_radius15mi.csv"

# ============================================================
# SETTINGS
# ============================================================
RADIUS_MILES        = 15.0   # <-- changed to 15 miles
TIME_HORIZON_YEARS  = 1.0

# Texas sanity bounds (prevents Africa/ocean)
TX_LAT_MIN, TX_LAT_MAX = 25.0, 37.5
TX_LON_MIN, TX_LON_MAX = -107.5, -93.0

# ============================================================
# Helpers: Correct NBI lat/long conversion
# ============================================================
def nbi_lat_to_dd(x):
    """LAT_016: DDMMSSss -> decimal degrees. If already decimal (-90..90), keep."""
    if pd.isna(x):
        return np.nan
    try:
        v = float(x)
    except:
        return np.nan

    if -90.0 <= v <= 90.0:
        return v

    try:
        v = int(abs(v))
        d = v // 1_000_000
        m = (v // 10_000) % 100
        s = (v % 10_000) / 100.0
        return d + m/60.0 + s/3600.0
    except:
        return np.nan

def nbi_lon_to_dd(x):
    """LONG_017: DDDMMSSss -> decimal degrees (West negative). If already decimal, keep."""
    if pd.isna(x):
        return np.nan
    try:
        v = float(x)
    except:
        return np.nan

    if -180.0 <= v <= 180.0:
        return v if v <= 0 else -v

    try:
        v = int(abs(v))
        d = v // 1_000_000
        m = (v // 10_000) % 100
        s = (v % 10_000) / 100.0
        lon = d + m/60.0 + s/3600.0
        return -lon
    except:
        return np.nan

def miles_to_radians(miles):
    return miles / 3958.7613  # Earth radius in miles

def format_coord(lat, lon, decimals=6):
    if pd.isna(lat) or pd.isna(lon):
        return ""
    return f"{lat:.{decimals}f}, {lon:.{decimals}f}"

# ============================================================
# 1) LOAD FILES
# ============================================================
bridges = pd.read_csv(BRIDGE_FILE, low_memory=False, dtype={"STRUCTURE_NUMBER_008": str})
storms  = pd.read_csv(IBTRACS_FILE, low_memory=False)

# ============================================================
# 2) BRIDGE COORDINATES + TEXAS SANITY FILTER
# ============================================================
if {"LAT_016", "LONG_017"}.issubset(bridges.columns):
    bridges["lat"] = bridges["LAT_016"].apply(nbi_lat_to_dd)
    bridges["lon"] = bridges["LONG_017"].apply(nbi_lon_to_dd)
elif {"LATITUDE", "LONGITUDE"}.issubset(bridges.columns):
    bridges["lat"] = pd.to_numeric(bridges["LATITUDE"], errors="coerce")
    bridges["lon"] = pd.to_numeric(bridges["LONGITUDE"], errors="coerce")
else:
    raise ValueError("Bridge coords not found. Expected LAT_016/LONG_017 or LATITUDE/LONGITUDE.")

# Texas sanity box: anything outside -> NaN
tx_ok = bridges["lat"].between(TX_LAT_MIN, TX_LAT_MAX) & bridges["lon"].between(TX_LON_MIN, TX_LON_MAX)
bridges.loc[~tx_ok, ["lat", "lon"]] = np.nan

# Coordinate string
bridges["coordinates (lat, lon)"] = [
    format_coord(a, b, 6) for a, b in zip(bridges["lat"], bridges["lon"])
]

# Drop bridges without valid TX coords
bridges = bridges.dropna(subset=["lat", "lon"]).copy()
print(f"Bridges with valid TX coordinates: {len(bridges):,}")

# ============================================================
# 3) PREP STORM DATA (IBTrACS)
# ============================================================
storms = storms.rename(columns={
    "Storm Identifier": "storm_id",
    "Calendar Year": "year",
    "Geograpic Latitude": "lat",
    "Geographic Longitude": "lon",
    "Maximum Sustained Wind (knots)": "wind_kt",
})

storms["lat"] = pd.to_numeric(storms["lat"], errors="coerce")
storms["lon"] = pd.to_numeric(storms["lon"], errors="coerce")
storms["wind_kt"] = pd.to_numeric(storms["wind_kt"], errors="coerce")
storms["year"] = pd.to_numeric(storms["year"], errors="coerce")
storms = storms.dropna(subset=["storm_id", "year", "lat", "lon", "wind_kt"]).copy()

# Wind mph
storms["wind_mph"] = storms["wind_kt"] * 1.15078

min_year = int(storms["year"].min())
max_year = int(storms["year"].max())
years_covered = max(1, (max_year - min_year + 1))
print(f"IBTrACS years covered: {min_year}–{max_year} (Y={years_covered})")

# ============================================================
# 4) SSHWS CATEGORY
# ============================================================
CAT_THRESH = {1: 74.0, 2: 96.0, 3: 111.0, 4: 130.0, 5: 157.0}

def wind_to_cat(mph):
    if mph >= CAT_THRESH[5]: return 5
    if mph >= CAT_THRESH[4]: return 4
    if mph >= CAT_THRESH[3]: return 3
    if mph >= CAT_THRESH[2]: return 2
    if mph >= CAT_THRESH[1]: return 1
    return 0

storms["cat"] = storms["wind_mph"].apply(wind_to_cat)

# ============================================================
# 5) BALLTREE FOR STORMS (spatial query within radius)
# ============================================================
storm_points_rad = np.radians(storms[["lat", "lon"]].to_numpy(dtype=float))
tree = BallTree(storm_points_rad, metric="haversine")
radius_rad = miles_to_radians(RADIUS_MILES)

storm_id_arr  = storms["storm_id"].to_numpy()
storm_cat_arr = storms["cat"].to_numpy()

# ============================================================
# 6) HAZARD COUNTS + POISSON PROBABILITY (FDOT-style form)
#    - Count UNIQUE storms within radius, using each storm's max category near bridge
#    - λ = N / years_covered
#    - p(t) = 1 - exp(-λ t)
# ============================================================
bridge_points_rad = np.radians(bridges[["lat", "lon"]].to_numpy(dtype=float))
neighbors = tree.query_radius(bridge_points_rad, r=radius_rad)

N_by_cat = {g: np.zeros(len(bridges), dtype=int) for g in range(1, 6)}
p_by_cat = {g: np.zeros(len(bridges), dtype=float) for g in range(1, 6)}

for i, idxs in enumerate(neighbors):
    if idxs.size == 0:
        continue

    sub_ids  = storm_id_arr[idxs]
    sub_cats = storm_cat_arr[idxs]

    # per-storm maximum category near the bridge (dedupe storm_id)
    maxcat = {}
    for sid, c in zip(sub_ids, sub_cats):
        if sid not in maxcat or c > maxcat[sid]:
            maxcat[sid] = c

    maxcats = np.array(list(maxcat.values()), dtype=int)

    for g in range(1, 6):
        n = int(np.sum(maxcats >= g))
        N_by_cat[g][i] = n
        lam = n / years_covered
        p_by_cat[g][i] = 1.0 - np.exp(-lam * TIME_HORIZON_YEARS)

for g in range(1, 6):
    bridges[f"N_storm_cat{g}"] = N_by_cat[g]
    bridges[f"p_cat{g}"] = p_by_cat[g]

# optional: keep record length in output
bridges["years_covered_ibtracs"] = years_covered
bridges["radius_miles_used"] = RADIUS_MILES

# ============================================================
# 7) VULNERABILITY MULTIPLIERS (NBI-based proxies)
# ============================================================
def v_des_from_43a(x):
    try:
        x = int(float(x))
    except:
        return 1.0
    if x in [1, 2]:      # concrete
        s = 2
    elif x in [3, 4]:    # steel
        s = 3
    elif x == 5:         # timber
        s = 5
    else:
        s = 3
    return (s + 1.0) / 3.0

if "STRUCTURE_KIND_043A" in bridges.columns:
    bridges["V_des_hurr"] = bridges["STRUCTURE_KIND_043A"].apply(v_des_from_43a)
else:
    bridges["V_des_hurr"] = 1.0

def v_scour_from_113(code):
    if pd.isna(code):
        return 1.0
    c = str(code).strip().upper()
    high_bin = {"0","1","2","3","4","6","U"}
    low_bin  = {"5","7","8","9","N","T"}
    if c in high_bin: return 1.2
    if c in low_bin:  return 0.4
    return 1.0

scour_col = None
for c in ["SCOUR_CRITICAL_113", "SCOUR_CRITICAL_113A", "SCOUR_113", "NBI_113"]:
    if c in bridges.columns:
        scour_col = c
        break
if scour_col is None:
    cand = [c for c in bridges.columns if "113" in c.upper()]
    scour_col = cand[0] if cand else None

bridges["V_scour_hurr"] = bridges[scour_col].apply(v_scour_from_113) if scour_col else 1.0

BASE_YEAR = 1990
DENOM = 35.0
def v_age(year_built):
    y = pd.to_numeric(year_built, errors="coerce")
    if pd.isna(y):
        return 1.0
    return max(1.0, 1.0 + (BASE_YEAR - float(y)) / DENOM)

bridges["V_age_hurr"] = bridges["YEAR_BUILT_027"].apply(v_age) if "YEAR_BUILT_027" in bridges.columns else 1.0

# Element vulnerability proxy from condition ratings
cond_cols = [c for c in ["DECK_COND_058", "SUPERSTRUCTURE_COND_059", "SUBSTRUCTURE_COND_060"] if c in bridges.columns]

def rating_to_s0(r):
    r = pd.to_numeric(r, errors="coerce")
    if pd.isna(r): return 2.5
    s = (9.0 - float(r)) * (5.0 / 9.0)
    return float(np.clip(s, 0.0, 5.0))

if cond_cols:
    base_s = bridges[cond_cols].applymap(rating_to_s0).mean(axis=1)
else:
    base_s = pd.Series(np.full(len(bridges), 2.5), index=bridges.index)

for g in range(1, 6):
    sg = np.clip(base_s + 0.5*(g-1), 0.0, 5.0)
    bridges[f"V_elem_cat{g}"] = (sg + 1.0) / 3.0

# ============================================================
# 8) RISK INDEX (category-summed, FDOT-style form)
# ============================================================
bridges["Risk_hurricane"] = 0.0
for g in range(1, 6):
    bridges["Risk_hurricane"] += (
        bridges[f"p_cat{g}"] *
        bridges[f"V_elem_cat{g}"] *
        bridges["V_des_hurr"] *
        bridges["V_scour_hurr"] *
        bridges["V_age_hurr"]
    )

# ============================================================
# 9) SAVE FINAL CSV (includes storm counts + probabilities)
# ============================================================
base_cols = [
    "STRUCTURE_NUMBER_008",
    "LOCATION_009",
    "lat",
    "lon",
    "YEAR_BUILT_027",
    "V_des_hurr",
    "V_scour_hurr",
    "V_age_hurr",
    "Risk_hurricane",
    "coordinates (lat, lon)",
    "years_covered_ibtracs",
    "radius_miles_used",
]

storm_cols = []
for g in range(1, 6):
    storm_cols += [f"N_storm_cat{g}", f"p_cat{g}"]

out_cols = base_cols + storm_cols
out_cols = [c for c in out_cols if c in bridges.columns]

out = bridges[out_cols].copy().sort_values("Risk_hurricane", ascending=False)
out.to_csv(OUTPUT_FILE, index=False)

print(f"\n✅ Saved ranked results: {OUTPUT_FILE}")
print(out.head(10).to_string(index=False))

blank_coords = (out["coordinates (lat, lon)"].astype(str).str.len() == 0).sum() if "coordinates (lat, lon)" in out.columns else 0
print(f"\nBridges in output: {len(out):,} | Blank coordinate strings: {blank_coords:,}")

## other truck collision

In [ ]:
# ============================================================
# TxDOT / FDOT "Other Truck Collisions" Risk Index
# FINAL ROBUST VERSION
#
# USES:
#   - Crash_ID for deduplication
#   - crash latitude/longitude for nearest-bridge matching
#   - auto-detects the crash-type/code column and keeps ONLY code 42
#
# DOES NOT REQUIRE the exact header name "Bridge_Detail_ID"
# ============================================================

import os
import warnings
import numpy as np
import pandas as pd
from sklearn.neighbors import BallTree

warnings.filterwarnings("ignore", category=FutureWarning)

# ============================================================
# USER SETTINGS
# ============================================================
BRIDGE_FILE = "TX25.csv"
CRASH_FILE  = "0-7239_Crashes that involve bridge_Zhe.csv"

OBS_YEARS = 10.0
MAX_MATCH_DISTANCE_M = 120.0
TARGET_CODE = 42

OUTPUT_MATCHED = "matched_other_truck_collision_code42.csv"
OUTPUT_ALL = "txdot_other_truck_collision_risk_all_bridges.csv"
OUTPUT_TOP100 = "txdot_other_truck_collision_risk_top100.csv"

# ============================================================
# HELPERS
# ============================================================
def find_first_existing(df, candidates, required=False):
    cols_map = {str(c).strip().lower(): c for c in df.columns}
    for cand in candidates:
        key = cand.strip().lower()
        if key in cols_map:
            return cols_map[key]
    if required:
        raise KeyError(f"Could not find any of these columns: {candidates}")
    return None


def clean_num(s):
    return pd.to_numeric(s, errors="coerce")


def nbi_lat_to_dd(x):
    if pd.isna(x):
        return np.nan
    try:
        v = float(x)
    except Exception:
        return np.nan

    if -90 <= v <= 90:
        return v

    s = str(int(abs(v))).zfill(8)
    deg = int(s[:2])
    minute = int(s[2:4])
    sec = int(s[4:6])
    sec_dec = int(s[6:8]) / 100.0
    dd = deg + minute / 60.0 + (sec + sec_dec) / 3600.0
    return dd if v >= 0 else -dd


def nbi_lon_to_dd(x):
    if pd.isna(x):
        return np.nan
    try:
        v = float(x)
    except Exception:
        return np.nan

    if -180 <= v <= 180:
        return v

    s = str(int(abs(v))).zfill(9)
    deg = int(s[:3])
    minute = int(s[3:5])
    sec = int(s[5:7])
    sec_dec = int(s[7:9]) / 100.0
    dd = deg + minute / 60.0 + (sec + sec_dec) / 3600.0

    # Texas longitude should be west/negative
    return -dd if v >= 0 else dd


def read_crash_csv_safely(path):
    """
    TxDOT exports sometimes have metadata rows before the real header.
    """
    for skip in [12, 0, 10, 11, 13]:
        try:
            df = pd.read_csv(path, skiprows=skip, low_memory=False)
            lower_cols = [str(c).strip().lower() for c in df.columns]
            if any(c in lower_cols for c in ["crash_id", "latitude", "longitude", "rpt_latitude", "rpt_longitude"]):
                print(f"[INFO] Crash file loaded with skiprows={skip}")
                return df
        except Exception:
            pass

    df = pd.read_csv(path, low_memory=False)
    print("[INFO] Crash file loaded with fallback skiprows=0")
    return df


def detect_code42_column(df):
    """
    Try to detect the crash-type / bridge-detail code column automatically.

    Strategy:
    1. Prefer columns whose names look like bridge/detail/object/impact.
    2. Among numeric-like columns, find one containing values from {40,41,42,43,44}.
    """
    target_set = {40, 41, 42, 43, 44}
    candidates = []

    for col in df.columns:
        s = pd.to_numeric(df[col], errors="coerce")
        non_na = s.dropna()

        if len(non_na) == 0:
            continue

        uniq = set(non_na.astype(int).unique().tolist())

        # Need at least 42 present
        if 42 not in uniq:
            continue

        overlap = len(uniq.intersection(target_set))
        if overlap == 0:
            continue

        name = str(col).lower()
        name_score = 0
        for token in ["bridge", "detail", "object", "struck", "impact", "harmful", "relation"]:
            if token in name:
                name_score += 1

        candidates.append({
            "col": col,
            "overlap": overlap,
            "name_score": name_score,
            "sample_values": sorted(list(uniq.intersection(target_set)))
        })

    if not candidates:
        return None, []

    candidates = sorted(
        candidates,
        key=lambda x: (x["name_score"], x["overlap"]),
        reverse=True
    )
    return candidates[0]["col"], candidates


def get_design_vulnerability(kind, typ):
    try:
        kind = int(kind) if pd.notna(kind) else -1
    except Exception:
        kind = -1

    try:
        typ = int(typ) if pd.notna(typ) else -1
    except Exception:
        typ = -1

    if kind in {1, 2}:
        return 1.10
    elif kind in {3, 4}:
        return 1.05
    elif kind in {5, 6}:
        return 1.15
    elif kind in {7, 8}:
        return 1.20
    else:
        return 1.08


def get_age_vulnerability(year_built, ref_year=1990, k=35):
    y = pd.to_numeric(year_built, errors="coerce")
    if pd.isna(y):
        return 1.05
    return max(1.0, 1.0 + (ref_year - y) / k)


def get_element_vulnerability(row):
    vals = []
    for c in [
        "RAILINGS_036A",
        "APPR_RAIL_036C",
        "APPR_RAIL_END_036D",
        "SUPERSTRUCTURE_COND_059",
        "SUBSTRUCTURE_COND_060",
        "UNDCLRENCE_EVAL_069",
    ]:
        if c in row.index:
            v = pd.to_numeric(row[c], errors="coerce")
            if pd.notna(v):
                vals.append(v)

    if len(vals) == 0:
        return 1.10

    avg = np.mean(vals)

    if avg >= 8:
        score = 0
    elif avg >= 6:
        score = 1
    elif avg >= 4:
        score = 2
    else:
        score = 3

    return 1.0 + score / 3.0


# ============================================================
# 1) LOAD BRIDGE INVENTORY
# ============================================================
bridges = pd.read_csv(BRIDGE_FILE, low_memory=False)

BRIDGE_ID_COL = "STRUCTURE_NUMBER_008"
BRIDGE_LAT_COL = "LAT_016"
BRIDGE_LON_COL = "LONG_017"
YEAR_BUILT_COL = "YEAR_BUILT_027"
ADT_COL = "ADT_029"
TRUCK_PCT_COL = "PERCENT_ADT_TRUCK_109"
KIND_COL = "STRUCTURE_KIND_043A"
TYPE_COL = "STRUCTURE_TYPE_043B"

required_bridge_cols = [BRIDGE_ID_COL, BRIDGE_LAT_COL, BRIDGE_LON_COL, YEAR_BUILT_COL]
missing_bridge_cols = [c for c in required_bridge_cols if c not in bridges.columns]
if missing_bridge_cols:
    raise KeyError(f"Bridge CSV is missing required columns: {missing_bridge_cols}")

bridges["bridge_id"] = bridges[BRIDGE_ID_COL].astype(str).str.strip()
bridges["bridge_lat"] = bridges[BRIDGE_LAT_COL].apply(nbi_lat_to_dd)
bridges["bridge_lon"] = bridges[BRIDGE_LON_COL].apply(nbi_lon_to_dd)

bridges = bridges[
    bridges["bridge_lat"].between(25.0, 37.5, inclusive="both") &
    bridges["bridge_lon"].between(-107.5, -93.0, inclusive="both")
].copy()

bridges = bridges.dropna(subset=["bridge_id", "bridge_lat", "bridge_lon"]).reset_index(drop=True)

bridges["ADT"] = clean_num(bridges[ADT_COL]) if ADT_COL in bridges.columns else np.nan
bridges["truck_pct"] = clean_num(bridges[TRUCK_PCT_COL]) if TRUCK_PCT_COL in bridges.columns else np.nan

bridges["V_age_truck"] = bridges[YEAR_BUILT_COL].apply(get_age_vulnerability)
bridges["V_des_truck"] = bridges.apply(
    lambda r: get_design_vulnerability(
        r[KIND_COL] if KIND_COL in bridges.columns else np.nan,
        r[TYPE_COL] if TYPE_COL in bridges.columns else np.nan
    ),
    axis=1
)
bridges["V_elem_truck"] = bridges.apply(get_element_vulnerability, axis=1)

print(f"[INFO] Bridges loaded: {len(bridges):,}")

# ============================================================
# 2) LOAD CRASH FILE
# ============================================================
if not os.path.exists(CRASH_FILE):
    raise FileNotFoundError(f"Crash file not found: {CRASH_FILE}")

crashes = read_crash_csv_safely(CRASH_FILE)

print("\n[INFO] Crash columns detected:")
print(list(crashes.columns))

CRASH_ID_CANDIDATES = ["Crash_ID", "CRASH_ID", "crash_id"]
LAT_CANDIDATES = ["Rpt_Latitude", "Latitude", "RPT_LATITUDE", "LATITUDE", "latitude"]
LON_CANDIDATES = ["Rpt_Longitude", "Longitude", "RPT_LONGITUDE", "LONGITUDE", "longitude"]

crash_id_col = find_first_existing(crashes, CRASH_ID_CANDIDATES, required=False)
lat_col = find_first_existing(crashes, LAT_CANDIDATES, required=True)
lon_col = find_first_existing(crashes, LON_CANDIDATES, required=True)

print("\n[INFO] Using crash columns:")
print("  Crash ID  :", crash_id_col)
print("  Latitude  :", lat_col)
print("  Longitude :", lon_col)

# Use Crash_ID for deduplication
if crash_id_col is not None:
    crashes["crash_id"] = crashes[crash_id_col].astype(str).str.strip()
else:
    crashes["crash_id"] = crashes.index.astype(str)

crashes["crash_lat"] = clean_num(crashes[lat_col])
crashes["crash_lon"] = clean_num(crashes[lon_col])

crashes = crashes.dropna(subset=["crash_lat", "crash_lon"]).copy()

crashes = crashes[
    crashes["crash_lat"].between(25.0, 37.5, inclusive="both") &
    crashes["crash_lon"].between(-107.5, -93.0, inclusive="both")
].copy()

crashes = crashes.drop_duplicates(subset=["crash_id"]).reset_index(drop=True)
print(f"[INFO] Unique crash rows after deduplication: {len(crashes):,}")

# ============================================================
# 3) AUTO-DETECT CODE COLUMN, KEEP ONLY CODE 42
# ============================================================
code_col, code_candidates = detect_code42_column(crashes)

if code_col is None:
    print("\n[ERROR] Could not auto-detect a column containing code 42.")
    print("Potential fix: inspect the crash CSV headers and values.")
    print("You can run:")
    print("    print(crashes.columns.tolist())")
    raise ValueError("No suitable code-42 column detected in crash file.")

print(f"\n[INFO] Auto-detected code column: {code_col}")
print("[INFO] Top candidate columns:")
for item in code_candidates[:10]:
    print(f"   - {item['col']} | codes found: {item['sample_values']} | name_score={item['name_score']}")

crashes["code42_col"] = pd.to_numeric(crashes[code_col], errors="coerce")

before_filter = len(crashes)
crashes = crashes[crashes["code42_col"] == TARGET_CODE].copy()
print(f"[INFO] Code 42 filter kept {len(crashes):,} of {before_filter:,} deduplicated crashes")

if len(crashes) == 0:
    raise ValueError("No code-42 crashes remain after filtering.")

# ============================================================
# 4) SPATIAL MATCH TO NEAREST BRIDGE
# ============================================================
bridges["lat_rad"] = np.radians(bridges["bridge_lat"])
bridges["lon_rad"] = np.radians(bridges["bridge_lon"])
bridge_coords = np.c_[bridges["lat_rad"].values, bridges["lon_rad"].values]

tree = BallTree(bridge_coords, metric="haversine")

crashes["lat_rad"] = np.radians(crashes["crash_lat"])
crashes["lon_rad"] = np.radians(crashes["crash_lon"])
crash_coords = np.c_[crashes["lat_rad"].values, crashes["lon_rad"].values]

dist_rad, ind = tree.query(crash_coords, k=1)
earth_radius_m = 6371000.0

crashes["nearest_bridge_idx"] = ind.flatten()
crashes["match_distance_m"] = dist_rad.flatten() * earth_radius_m

matched = crashes[crashes["match_distance_m"] <= MAX_MATCH_DISTANCE_M].copy()

if len(matched) == 0:
    raise ValueError(
        f"No code-42 crashes matched within {MAX_MATCH_DISTANCE_M} m. "
        f"Try increasing MAX_MATCH_DISTANCE_M to 150 or 200."
    )

matched["bridge_id"] = bridges.iloc[matched["nearest_bridge_idx"]][BRIDGE_ID_COL].values
matched["bridge_lat_match"] = bridges.iloc[matched["nearest_bridge_idx"]]["bridge_lat"].values
matched["bridge_lon_match"] = bridges.iloc[matched["nearest_bridge_idx"]]["bridge_lon"].values

print(f"[INFO] Matched code-42 crashes: {len(matched):,}")

matched.to_csv(OUTPUT_MATCHED, index=False)

# ============================================================
# 5) COUNT EVENTS PER BRIDGE
# ============================================================
counts = matched.groupby("bridge_id").size().reset_index(name="N_i")
counts["lambda_i"] = counts["N_i"] / OBS_YEARS
counts["p_truck"] = 1.0 - np.exp(-counts["lambda_i"])

# ============================================================
# 6) MERGE + RISK INDEX
# ============================================================
risk = bridges.merge(counts, on="bridge_id", how="left")

risk["N_i"] = risk["N_i"].fillna(0).astype(int)
risk["lambda_i"] = risk["lambda_i"].fillna(0.0)
risk["p_truck"] = risk["p_truck"].fillna(0.0)

risk["Risk_truck"] = (
    risk["p_truck"] *
    risk["V_elem_truck"] *
    risk["V_age_truck"] *
    risk["V_des_truck"]
)

risk["truck_pct_frac"] = risk["truck_pct"] / 100.0
risk["truck_pct_frac"] = risk["truck_pct_frac"].clip(lower=0, upper=1)

valid_exposure = (risk["ADT"] * risk["truck_pct_frac"]).replace(0, np.nan)
ref_exposure = valid_exposure.median()

if pd.notna(ref_exposure) and ref_exposure > 0:
    risk["exposure_factor"] = (risk["ADT"] * risk["truck_pct_frac"]) / ref_exposure
    risk["exposure_factor"] = risk["exposure_factor"].clip(lower=0.25, upper=4.0)
else:
    risk["exposure_factor"] = 1.0

risk["Risk_truck_exposure_adj"] = risk["Risk_truck"] * risk["exposure_factor"]

# ============================================================
# 7) SAVE OUTPUTS
# ============================================================
keep_cols = [
    "bridge_id",
    "bridge_lat", "bridge_lon",
    YEAR_BUILT_COL,
    "ADT", "truck_pct",
    KIND_COL if KIND_COL in risk.columns else None,
    TYPE_COL if TYPE_COL in risk.columns else None,
    "N_i", "lambda_i", "p_truck",
    "V_elem_truck", "V_age_truck", "V_des_truck",
    "exposure_factor",
    "Risk_truck",
    "Risk_truck_exposure_adj",
]
keep_cols = [c for c in keep_cols if c is not None and c in risk.columns]

risk_sorted = risk.sort_values(
    by=["Risk_truck_exposure_adj", "Risk_truck", "N_i", "p_truck"],
    ascending=False
).reset_index(drop=True)

risk_sorted[keep_cols].to_csv(OUTPUT_ALL, index=False)
risk_sorted.head(100)[keep_cols].to_csv(OUTPUT_TOP100, index=False)

# ============================================================
# 8) SUMMARY
# ============================================================
print("\n============================================================")
print("DONE")
print("============================================================")
print(f"Matched crash file    : {OUTPUT_MATCHED}")
print(f"All bridges risk file : {OUTPUT_ALL}")
print(f"Top 100 risk file     : {OUTPUT_TOP100}")

print("\nTop 15 bridges:")
print(risk_sorted[keep_cols].head(15).to_string(index=False))

[INFO] Bridges loaded: 56,951
[INFO] Crash file loaded with skiprows=12

[INFO] Crash columns detected:
['Crash ID', 'Average Daily Traffic Amount', 'Average Daily Traffic Year', 'Bridge Detail', 'Case ID', 'City', 'Contributing Factors', 'County', 'Crash Date', 'Crash Death Count', 'Crash Month', 'Crash Non-Suspected Serious Injury Count', 'Crash Not Injured Count', 'Crash Number', 'Crash Possible Injury Count', 'Crash Severity', 'Crash Suspected Serious Injury Count', 'Crash Time', 'Crash Total Injury Count', 'Crash Unknown Injury Count', 'Crash Year', 'Date Arrived', 'Date Notified', 'Date Roadway Cleared', 'Date Scene Cleared', 'DFO', 'Direction of Traffic', 'Feature Crossed by Bridge', 'First Harmful Event', 'Highway Alpha Suffix', 'Highway Number', 'Highway System', 'Latitude', 'Left Curb Type', 'Left Shoulder Type', 'Left Shoulder Use', 'Left Shoulder Width', 'Light Condition', 'Longitude', 'Manner of Collision', 'Median Type', 'Median Width', 'Number of Lanes', 'Object Struck',